# Part 1 Risk Scoring

### Pipeline to parse all drugs under ATC group N05A (Antipsychotics)

In [ ]:
"""
Pipeline to parse all drugs under ATC group N05A (Antipsychotics)
from the WHO ATC/DDD Index (https://atcddd.fhi.no/atc_ddd_index/).

Strategy:
    1. Fetch the N05A index page.
    2. Extract the child subgroup codes
       (N05AA, N05AB, N05AC, N05AD, N05AE, N05AF, N05AG,
        N05AH, N05AL, N05AN, N05AX).
    3. Visit each subgroup page and parse the drug-level rows
       (ATC code + name, and where available DDD / unit / route / note).
    4. Return a flat list of drug dicts.
    5. Save the results to a CSV file.

Notes:
    - N05AN covers lithium (used in bipolar disorder).
    - Reserpine is NOT in N05A; it is classified under C02
      (antihypertensives), so it will not appear in these results.
    - Some atypicals (e.g. aripiprazole, risperidone, lurasidone,
      ziprasidone) live in N05AX ("Other antipsychotics").
"""

import csv
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://atcddd.fhi.no/atc_ddd_index/"
HEADERS = {"User-Agent": "Mozilla/5.0 (ATC-DDD-parser; research use)"}

# Column order used for the CSV output
CSV_FIELDS = ["atc_code", "name", "ddd", "unit", "route", "note", "subgroup"]


# ----------------------------------------------------------------------
# Low-level fetch
# ----------------------------------------------------------------------
def fetch(code: str, session: requests.Session) -> BeautifulSoup:
    """Fetch a single ATC code page and return parsed soup."""
    resp = session.get(BASE_URL, params={"code": code}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


# ----------------------------------------------------------------------
# Discover subgroup codes under a parent code
# ----------------------------------------------------------------------
def get_child_codes(soup: BeautifulSoup, parent_code: str) -> list[str]:
    """
    Extract immediate child ATC codes from a page.
    Children appear as links of the form '?code=N05AA'.
    """
    codes = []
    for a in soup.select("a[href*='code=']"):
        href = a["href"]
        code = href.split("code=")[-1].split("&")[0].strip()
        # keep codes that extend the parent (e.g. N05AA under N05A)
        if (
            code.startswith(parent_code)
            and code != parent_code
            and len(code) == len(parent_code) + 1   # one level deeper
            and code not in codes
        ):
            codes.append(code)
    return codes


# ----------------------------------------------------------------------
# Parse drug-level rows from a subgroup page
# ----------------------------------------------------------------------
def parse_drugs(soup: BeautifulSoup, subgroup_code: str) -> list[dict]:
    """
    On a subgroup (chemical subgroup) page the substances are listed in a
    table whose rows look like:
        ATC code | Name | DDD | Unit | Adm.R | Note
    The 7-character codes (e.g. N05AB03) identify chemical substances.
    """
    drugs = []
    for row in soup.select("table tr"):
        cells = [c.get_text(strip=True) for c in row.find_all("td")]
        if not cells:
            continue

        atc_code = cells[0]
        # A substance-level code is the subgroup code + 2 digits, e.g. N05AB03
        if not (atc_code.startswith(subgroup_code) and len(atc_code) == len(subgroup_code) + 2):
            continue

        drug = {
            "atc_code": atc_code,
            "name":     cells[1] if len(cells) > 1 else None,
            "ddd":      cells[2] if len(cells) > 2 and cells[2] else None,
            "unit":     cells[3] if len(cells) > 3 and cells[3] else None,
            "route":    cells[4] if len(cells) > 4 and cells[4] else None,
            "note":     cells[5] if len(cells) > 5 and cells[5] else None,
            "subgroup": subgroup_code,
        }
        # Some substances span multiple DDD/route rows; the name cell is empty
        # on continuation rows, so carry the previous name forward.
        if not drug["name"] and drugs:
            drug["name"] = drugs[-1]["name"]
        drugs.append(drug)
    return drugs


# ----------------------------------------------------------------------
# Orchestration
# ----------------------------------------------------------------------
def scrape_atc(parent_code: str = "N05A", polite_delay: float = 0.5) -> list[dict]:
    with requests.Session() as session:
        root = fetch(parent_code, session)
        subgroups = get_child_codes(root, parent_code)

        all_drugs = []
        for sub in subgroups:
            sub_soup = fetch(sub, session)
            all_drugs.extend(parse_drugs(sub_soup, sub))
            time.sleep(polite_delay)   # be courteous to the server
        return all_drugs


# ----------------------------------------------------------------------
# CSV export
# ----------------------------------------------------------------------
def save_to_csv(drugs: list[dict], filename: str = "atc_n05a_drugs.csv") -> None:
    """
    Write the parsed drug entries to a CSV file.

    Each dict in `drugs` is written as one row using the column order in
    CSV_FIELDS. Missing values (None) are written as empty cells.
    """
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        writer.writeheader()
        for d in drugs:
            # Replace None with "" so empty cells are clean in the CSV
            writer.writerow({k: (d.get(k) if d.get(k) is not None else "") for k in CSV_FIELDS})

    print(f"Saved {len(drugs)} rows to '{filename}'")


# ----------------------------------------------------------------------
if __name__ == "__main__":
    drugs = scrape_atc("N05A")
    print(f"Found {len(drugs)} drug entries under N05A (Antipsychotics)\n")
    for d in drugs:
        ddd = f"{d['ddd']} {d['unit']} ({d['route']})" if d["ddd"] else "no DDD"
        print(f"{d['atc_code']:<9} {d['name']:<30} {ddd}")

    # If you only want the plain list of drug names:
    names = sorted({d["name"] for d in drugs if d["name"]})
    print("\nUnique drug names:")
    print(names)

    # Save the full result set to CSV
    save_to_csv(drugs, "atc_n05a_drugs.csv")

Found 70 drug entries under N05A (Antipsychotics)

N05AA01   chlorpromazine                 0.3 g (O)
N05AA02   levomepromazine                0.3 g (O)
N05AA03   promazine                      0.3 g (O)
N05AA04   acepromazine                   0.1 g (O)
N05AA05   triflupromazine                0.1 g (O)
N05AA06   cyamemazine                    no DDD
N05AA07   chlorproethazine               no DDD
N05AB01   dixyrazine                     50 mg (O)
N05AB02   fluphenazine                   10 mg (O)
N05AB03   perphenazine                   30 mg (O)
N05AB04   prochlorperazine               0.1 g (O)
N05AB05   thiopropazate                  60 mg (O)
N05AB06   trifluoperazine                20 mg (O)
N05AB07   acetophenazine                 50 mg (O)
N05AB08   thioproperazine                75 mg (O)
N05AB09   butaperazine                   10 mg (O)
N05AB10   perazine                       0.1 g (O)
N05AC01   periciazine                    50 mg (O)
N05AC02   thioridazine               

In [ ]:
!cp "/content/atc_n05a_drugs.csv" "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_database/atc_n05a_drugs.csv"

### Ki Db

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_database" /content/ -r

### Loading twas

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity_cvs/dm" /content/ -r

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity_cvs/lipids" /content/ -r

In [ ]:
!pip install rapidfuzz tqdm openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 54.9 MB/s eta 0:00:00


# With weights

## Version 4 MILD EXPONENTIAL SCALING on -log10(p)

In [ ]:
# ======================================================================
# VERSION 3 (new):  N05A → Ki → DDD → TWAS pipeline
# TWAS INTEGRATION: MILD EXPONENTIAL SCALING on -log10(p)
#   twas_scale = exp(beta * -log10(p))     (beta ~ 0.03 - 0.06)
# Self-contained: run as a single Colab cell.
#   Requires: pip install pandas numpy rapidfuzz tqdm openpyxl
# ======================================================================

import os
import re
import glob
import datetime
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

# ======================================================================
# CONFIG  --  EDIT THESE PATHS
# ======================================================================
KI_DB_PATH     = "/content/Z_database/KiDatabase_2026-06-22.csv"
DRUGS_CSV_PATH = "/content/Z_database/atc_n05a_drugs.csv"

TWAS_DIRS = [
    "/content/lipids/ldl",
    "/content/lipids/hdl",
    "/content/lipids/logtg",
    "/content/lipids/nonhdl",
    "/content/lipids/tc",
]

# Distinct output root for the -log10(p) version
OUTPUT_DIR = "/content/pipeline_output/n05a_ki_twas_log_v3_logp"

FUZZY_THRESHOLD = 80
KI_AGGREGATION = "min"
STRONG_BINDER_KI_NM = 10.0
TWAS_P_SIGNIFICANT = 0.05
KI_IMPUTE_STRATEGY = "mean"
TWAS_IMPUTE_STRATEGY = "mean"
CSV_ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

# ---- Version 3 (logp_expo) scaling parameters ----
LOGP_BETA           = 0.045   # exponential rate on -log10(p) (tune 0.03 - 0.06)
MAX_NEGLOG10_P      = 25.0    # cap to prevent explosion from ultra-small p-values
SECONDARY_BOOST_FAC = 0.02    # tiny secondary drug-level boost (max|z|)

TWAS_SYMBOL_COL_CANDIDATES = [
    "gene_name", "genename", "gene_symbol", "symbol", "hgnc_symbol",
]
TWAS_ENSG_COL_CANDIDATES = [
    "gene", "ensembl_gene_id", "gene_id", "geneid", "ensembl", "id",
]
TWAS_Z_COL_CANDIDATES = [
    "twas.z", "twas_z", "zscore", "z_score", "zstat", "z", "zscore",
]
TWAS_P_COL_CANDIDATES = [
    "twas.p", "twas_p", "pvalue", "p_value", "pval", "p",
]


# ======================================================================
# Receptor (Ki DB label) -> official HGNC gene symbol
# ======================================================================
RECEPTOR_TO_GENE = {
    "5-HT1A": "HTR1A", "5HT1A": "HTR1A",
    "5-HT1B": "HTR1B", "5HT1B": "HTR1B",
    "5-HT1D": "HTR1D",
    "5-HT1E": "HTR1E",
    "5-HT1F": "HTR1F",
    "5-HT2A": "HTR2A", "5HT2A": "HTR2A",
    "5-HT2B": "HTR2B",
    "5-HT2C": "HTR2C", "5HT2C": "HTR2C",
    "5-HT3":  "HTR3A",
    "5-HT5A": "HTR5A",
    "5-HT6":  "HTR6",
    "5-HT7":  "HTR7",
    "D1": "DRD1", "D2": "DRD2", "D3": "DRD3", "D4": "DRD4", "D5": "DRD5",
    "DRD1": "DRD1", "DRD2": "DRD2", "DRD3": "DRD3", "DRD4": "DRD4", "DRD5": "DRD5",
    "alpha1A": "ADRA1A", "alpha1B": "ADRA1B", "alpha1D": "ADRA1D",
    "alpha2A": "ADRA2A", "alpha2B": "ADRA2B", "alpha2C": "ADRA2C",
    "alpha1":  "ADRA1A", "alpha2": "ADRA2A",
    "beta1": "ADRB1", "beta2": "ADRB2", "beta3": "ADRB3",
    "H1": "HRH1", "H2": "HRH2", "H3": "HRH3", "H4": "HRH4",
    "M1": "CHRM1", "M2": "CHRM2", "M3": "CHRM3", "M4": "CHRM4", "M5": "CHRM5",
    "Muscarinic": "CHRM1",
    "Sigma1": "SIGMAR1", "Sigma 1": "SIGMAR1",
    "SERT": "SLC6A4", "DAT": "SLC6A3", "NET": "SLC6A2",
}
RECEPTOR_TO_GENE = {k.strip().lower(): v for k, v in RECEPTOR_TO_GENE.items()}


# ======================================================================
# METABOLIC / DIABETES RISK WEIGHTS FOR ANTIPSYCHOTICS
# ======================================================================
METABOLIC_WEIGHTS = {
    "HRH1":    1.00,
    "HTR2C":   0.92,
    "CHRM3":   0.85,
    "HTR2A":   0.55,
    "ADRA1A":  0.48,
    "ADRA1B":  0.48,
    "HTR6":    0.42,
    "ADRA2A":  0.32,
    "ADRA2B":  0.30,
    "ADRA2C":  0.30,
    "CHRM1":   0.28,
    "CHRM4":   0.22,
    "CHRM5":   0.20,
    "HTR7":    0.25,
    "DRD3":    0.18,
    "DRD2":    0.12,
    "DRD4":    0.10,
    "HTR1A":   0.08,
    "ADRB1":   0.08,
    "ADRB2":   0.07,
    "SIGMAR1": 0.05,
    "SLC6A4":  0.06,
    "SLC6A2":  0.05,
    "SLC6A3":  0.04,
}


# ======================================================================
# Helpers
# ======================================================================
def _is_nan(x) -> bool:
    if x is None:
        return True
    try:
        return bool(np.isnan(x))
    except (TypeError, ValueError):
        return False


def read_csv_robust(path: str, **kwargs) -> pd.DataFrame:
    kwargs.setdefault("low_memory", False)
    last_err = None
    for enc in CSV_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", **kwargs)
    except Exception:
        raise last_err if last_err else RuntimeError(f"Could not read {path}")


def normalise_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip().lower())


def receptor_to_gene(receptor: str):
    if not isinstance(receptor, str):
        return None
    key = receptor.strip().lower()
    if key in RECEPTOR_TO_GENE:
        return RECEPTOR_TO_GENE[key]
    key2 = re.sub(r"[\s\-]", "", key)
    for k, v in RECEPTOR_TO_GENE.items():
        if re.sub(r"[\s\-]", "", k) == key2:
            return v
    return None


def parse_ki(value) -> float:
    if value is None:
        return np.nan
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"na", "nan", "nd", "n/a", "-"}:
        return np.nan
    s = s.replace(",", "")
    s = re.sub(r"^[><=~]+", "", s)
    m = re.search(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", s)
    return float(m.group()) if m else np.nan


def ddd_to_mg(ddd, unit) -> float:
    ddd_val = parse_ki(ddd)
    if _is_nan(ddd_val):
        return np.nan
    u = (str(unit).strip().lower() if unit is not None else "")
    factors = {"g": 1000.0, "mg": 1.0, "mcg": 0.001, "µg": 0.001, "ug": 0.001}
    factor = factors.get(u, 1.0)
    return ddd_val * factor


def aggregate_ki(values: pd.Series, how: str) -> float:
    vals = values.dropna()
    if vals.empty:
        return np.nan
    if how == "min":
        return float(vals.min())
    if how == "median":
        return float(vals.median())
    if how == "mean":
        return float(vals.mean())
    return float(vals.min())


def find_col(df: pd.DataFrame, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _strip_ensg_version(s) -> str:
    return re.sub(r"\.\d+$", "", str(s).strip().upper())


def _safe_label(path: str) -> str:
    base = os.path.basename(os.path.normpath(path))
    if not base:
        base = re.sub(r"[^A-Za-z0-9_.-]+", "_", path).strip("_")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", base).strip("_") or "twas"


# ======================================================================
# Recompute affinity-derived columns
# ======================================================================
def _recompute_affinity_columns(results: pd.DataFrame) -> pd.DataFrame:
    ki_nM  = results["Ki_nM"]
    ddd_mg = results["DDD_mg"]

    has_ki = ki_nM.notna() & (ki_nM > 0)
    results["inv_Ki"] = np.where(has_ki, 1.0 / ki_nM, np.nan)

    has_inv_ki = results["inv_Ki"].notna()
    has_ddd    = ddd_mg.notna() & (ddd_mg > 0)
    log_ddd    = np.log(ddd_mg.where(has_ddd) + 1.0)
    results["affinity_dose_score"] = np.where(
        has_inv_ki & has_ddd, results["inv_Ki"] * log_ddd,
        np.where(has_inv_ki, results["inv_Ki"], np.nan),
    )

    results["strong_binder"] = ki_nM.notna() & (ki_nM < STRONG_BINDER_KI_NM)
    return results


def drop_zero_score_rows(results: pd.DataFrame):
    if results.empty or "affinity_dose_score" not in results.columns:
        return results, 0
    zero_mask = results["affinity_dose_score"] == 0
    n_dropped = int(zero_mask.sum())
    if n_dropped:
        results = results.loc[~zero_mask].reset_index(drop=True)
        print(f"  [drop-zero] removed {n_dropped} row(s) with affinity_dose_score == 0.")
    else:
        print("  [drop-zero] no rows with affinity_dose_score == 0.")
    return results, n_dropped


def impute_missing_ki(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "ki_imputed" not in results.columns:
        results["ki_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown KI_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid = results.loc[results["Ki_nM"].notna(), ["receptor", "Ki_nM"]]
    if valid.empty:
        print("  [ki-impute] no Ki values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        per_receptor = valid.groupby("receptor")["Ki_nM"].mean()
        global_ref = float(valid["Ki_nM"].mean())
    else:
        per_receptor = valid.groupby("receptor")["Ki_nM"].median()
        global_ref = float(valid["Ki_nM"].median())
    need = results["Ki_nM"].isna() & results["receptor"].notna()
    n_need = int(need.sum())
    if n_need == 0:
        print(f"  [ki-impute] strategy='{strategy}': no missing Ki values to fill.")
        return results
    filled_ref = results.loc[need, "receptor"].map(per_receptor)
    filled_ref = filled_ref.fillna(global_ref)
    results.loc[need, "Ki_nM"] = filled_ref.values
    results.loc[need, "ki_imputed"] = True
    results = _recompute_affinity_columns(results)
    print(f"  [ki-impute] strategy='{strategy}': filled {n_need} missing Ki "
          f"value(s) (global fallback Ki = {global_ref:.3g} nM).")
    return results


def impute_missing_twas(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "twas_imputed" not in results.columns:
        results["twas_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown TWAS_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid_z = results.loc[results["twas_z"].notna(), "twas_z"]
    valid_p = results.loc[results["twas_p"].notna(), "twas_p"]
    if valid_z.empty and valid_p.empty:
        print("  [twas-impute] no TWAS values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        fill_z = float(valid_z.mean()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.mean()) if not valid_p.empty else np.nan
    else:
        fill_z = float(valid_z.median()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.median()) if not valid_p.empty else np.nan
    need_z = results["gene"].notna() & results["twas_z"].isna()
    need_p = results["gene"].notna() & results["twas_p"].isna()
    n_need = int((need_z | need_p).sum())
    if n_need == 0:
        print(f"  [twas-impute] strategy='{strategy}': no missing TWAS values to fill.")
        return results
    if not _is_nan(fill_z):
        results.loc[need_z, "twas_z"] = fill_z
    if not _is_nan(fill_p):
        results.loc[need_p, "twas_p"] = fill_p
    results.loc[need_z | need_p, "twas_imputed"] = True
    print(f"  [twas-impute] strategy='{strategy}': filled {n_need} gene row(s) "
          f"(twas_z={fill_z:.3f}, twas_p={fill_p:.3g}); "
          f"imputed rows kept NON-significant.")
    return results


# ======================================================================
# TWAS lookup
# ======================================================================
def _read_twas_file(fpath: str):
    for sep in [",", "\t", None]:
        for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
            try:
                df = pd.read_csv(
                    fpath, sep=sep, encoding=enc, engine="python",
                    low_memory=False, on_bad_lines="skip"
                )
                if df is not None and not df.empty and df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    try:
        df = pd.read_csv(fpath, engine="python", on_bad_lines="skip")
        if df is not None and not df.empty:
            return df
    except Exception:
        pass
    return None


def build_twas_lookup(twas_dir: str):
    lookup = {}
    files = glob.glob(os.path.join(twas_dir, "*.*"))
    if not files:
        print(f"  [warn] No files found in {twas_dir}")
        return lookup
    successful_files = 0
    skipped_files = []
    for fpath in files:
        try:
            df = _read_twas_file(fpath)
            if df is None or df.empty:
                skipped_files.append((os.path.basename(fpath), "unreadable/empty"))
                continue
            sym_col  = find_col(df, TWAS_SYMBOL_COL_CANDIDATES)
            ensg_col = find_col(df, TWAS_ENSG_COL_CANDIDATES)
            z_col    = find_col(df, TWAS_Z_COL_CANDIDATES)
            p_col    = find_col(df, TWAS_P_COL_CANDIDATES)
            if ensg_col is not None and ensg_col == sym_col:
                ensg_col = None
            if z_col is None or (sym_col is None and ensg_col is None):
                skipped_files.append((
                    os.path.basename(fpath),
                    f"missing cols (z={z_col}, sym={sym_col}, ensg={ensg_col}); "
                    f"had {list(df.columns)[:8]}"
                ))
                continue
            n = len(df)
            sym_vals  = (df[sym_col].astype(str).str.strip().str.upper()
                         if sym_col else pd.Series([""] * n, index=df.index))
            ensg_vals = (df[ensg_col].map(_strip_ensg_version)
                         if ensg_col else pd.Series([""] * n, index=df.index))
            z_vals = df[z_col].map(parse_ki)
            p_vals = (df[p_col].map(parse_ki)
                      if p_col else pd.Series([np.nan] * n, index=df.index))
            bad = {"NAN", "NA", "NONE", ""}
            for i in df.index:
                gene = sym_vals[i]
                if gene in bad:
                    gene = ensg_vals[i]
                if gene in bad:
                    continue
                z = z_vals[i]
                p = p_vals[i]
                if _is_nan(z) and _is_nan(p):
                    continue
                prev = lookup.get(gene)
                if prev is None:
                    lookup[gene] = {"twas_z": z, "twas_p": p}
                else:
                    prev_p = prev.get("twas_p", np.nan)
                    if not _is_nan(p) and (_is_nan(prev_p) or p < prev_p):
                        lookup[gene] = {"twas_z": z, "twas_p": p}
            successful_files += 1
        except Exception as e:
            skipped_files.append((os.path.basename(fpath), f"error: {e}"))
            continue
    print(f"  Pre-loaded TWAS data for {len(lookup):,} genes "
          f"from {successful_files}/{len(files)} file(s)")
    if skipped_files:
        print(f"  [warn] {len(skipped_files)} file(s) skipped:")
        for name, reason in skipped_files[:8]:
            print(f"         - {name}: {reason}")
    return lookup


def get_twas_for_gene(gene: str, lookup):
    if gene is None or lookup is None:
        return None
    return lookup.get(str(gene).strip().upper())


# ======================================================================
# Metabolic risk score  --  VERSION 3: EXPONENTIAL scaling on -log10(p)
# ======================================================================
def calculate_metabolic_risk_score(
    df: pd.DataFrame,
    weights: dict = None,
    score_transform: str = "log1p",
    twas_boost: bool = True,
    twas_scaling: str = "logp_expo",     # "logp_expo" | "logp_linear" | "z_expo" | "z_linear" | "none"
    beta: float = 0.045,                 # exponential rate for logp_expo / z_expo
    gamma: float = 0.12,                 # linear factor for logp_linear
    twas_sig_bonus: float = 0.10,
    normalize: bool = True,
    reference_drug: str = "chlorpromazine",
    missing_gene_strategy: str = "mean_abs_z",
    max_neglog10_p: float = 25.0,        # cap to prevent explosion from ultra-small p
    secondary_boost_factor: float = 0.02,
) -> pd.DataFrame:
    """
    Per-drug Metabolic Risk Score for antipsychotics.

    TWAS INTEGRATION -- VERSION 3 (Mild exponential scaling on -log10(p)):
      Each receptor's weighted_contribution is multiplied by
      twas_scale = exp(beta * -log10(p)) BEFORE aggregation. This gives
      stronger emphasis to highly significant TWAS hits while remaining
      stable. -log10(p) is capped at `max_neglog10_p`. base_score is
      recomputed from the TWAS-scaled contributions; a tiny secondary
      drug-level boost (secondary_boost_factor * max|z|) is retained.

    twas_scaling options:
        "logp_expo"   : exp(beta  * -log10(p))     <- Version 3 default
        "logp_linear" : 1 + gamma * -log10(p)
        "z_expo"      : exp(beta  * |z|)           (v2 behaviour)
        "z_linear"    : 1 + beta  * |z|            (v1 behaviour)
        "none"        : no per-receptor TWAS scaling
    """
    if weights is None:
        weights = METABOLIC_WEIGHTS

    df = df.copy()
    mask = df["gene"].notna() & df["affinity_dose_score"].notna()
    metab_mask = mask & df["gene"].isin(weights.keys())

    # ---- report TWAS coverage among scored metabolic receptors ----
    n_metab = int(metab_mask.sum())
    uses_p = twas_scaling in ("logp_expo", "logp_linear")
    cov_col = "twas_p" if uses_p else "twas_z"
    n_metab_twas = int((metab_mask & df[cov_col].notna()).sum())
    if twas_boost and n_metab > 0:
        cov = n_metab_twas / n_metab
        if cov < 0.5:
            print(f"  [note] TWAS coverage is low: only {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS {cov_col}. "
                  f"Per-receptor scaling will be partial.")
        else:
            print(f"  [note] TWAS coverage: {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS {cov_col}.")

    if score_transform == "log1p":
        affinity = np.log1p(df.loc[mask, "affinity_dose_score"].clip(lower=0))
    elif score_transform == "none":
        affinity = df.loc[mask, "affinity_dose_score"]
    else:
        raise ValueError(f"Unknown score_transform: {score_transform!r}")

    df.loc[mask, "receptor_weight"] = df.loc[mask, "gene"].map(weights).fillna(0.0)
    df.loc[mask, "weighted_contribution"] = affinity * df.loc[mask, "receptor_weight"]

    drug_scores = (
        df.groupby("drug", dropna=False)["weighted_contribution"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_contribution": "base_score"})
    )

    # ==================================================================
    # TWAS integration - Version 3: exponential on -log10(p)
    # ==================================================================
    if twas_boost and twas_scaling != "none":
        df_filled = df.copy()

        # --- Fairness imputation for missing metabolic genes ---
        if missing_gene_strategy == "mean_abs_z":
            if uses_p:
                # impute missing metabolic p-values via the mean -log10(p)
                avail_p = df_filled.loc[
                    metab_mask & df_filled["twas_p"].notna(), "twas_p"
                ].clip(lower=1e-300)
                if len(avail_p) > 0:
                    mean_neglogp = float((-np.log10(avail_p)).mean())
                else:
                    mean_neglogp = 0.0
                impute_mask = (
                    mask
                    & df_filled["twas_p"].isna()
                    & df_filled["gene"].isin(weights.keys())
                )
                n_imputed = int(impute_mask.sum())
                if n_imputed > 0:
                    df_filled.loc[impute_mask, "twas_p"] = 10.0 ** (-mean_neglogp)
                    print(f"  [fairness] imputed {n_imputed} missing metabolic "
                          f"p-value(s) with mean -log10(p) = {mean_neglogp:.3f}")
            else:
                available_z = df_filled.loc[
                    metab_mask & df_filled["twas_z"].notna(), "twas_z"
                ].abs()
                mean_abs_z = float(available_z.mean()) if len(available_z) > 0 else 0.0
                impute_mask = (
                    mask
                    & df_filled["twas_z"].isna()
                    & df_filled["gene"].isin(weights.keys())
                )
                n_imputed = int(impute_mask.sum())
                if n_imputed > 0:
                    df_filled.loc[impute_mask, "twas_z"] = mean_abs_z
                    print(f"  [fairness] imputed {n_imputed} missing metabolic-gene "
                          f"row(s) with mean |twas_z| = {mean_abs_z:.3f}")
        elif missing_gene_strategy == "zero":
            print("  [fairness] missing_gene_strategy='zero' "
                  "(no TWAS boost for missing genes)")
        else:
            raise ValueError(
                f"Unknown missing_gene_strategy: {missing_gene_strategy!r} "
                f"(expected 'mean_abs_z' or 'zero')"
            )

        # --- Per-receptor TWAS scaling ---
        if twas_scaling == "logp_expo":
            p_safe = df_filled.loc[mask, "twas_p"].clip(lower=1e-300)
            neglog10_p = (-np.log10(p_safe)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = np.exp(beta * neglog10_p)
            print(f"  [TWAS-logp] exponential scaling on -log10(p) active "
                  f"(beta={beta}, cap={max_neglog10_p}). "
                  f"Secondary drug-level boost factor = {secondary_boost_factor}")
        elif twas_scaling == "logp_linear":
            p_safe = df_filled.loc[mask, "twas_p"].clip(lower=1e-300)
            neglog10_p = (-np.log10(p_safe)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = 1.0 + gamma * neglog10_p
            print(f"  [TWAS-logp] linear scaling on -log10(p) active "
                  f"(gamma={gamma}, cap={max_neglog10_p}).")
        elif twas_scaling == "z_expo":
            twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = np.exp(beta * twas_abs)
            print(f"  [TWAS-z] exponential scaling on |z| active (beta={beta}).")
        elif twas_scaling == "z_linear":
            twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = 1.0 + beta * twas_abs
            print(f"  [TWAS-z] linear scaling on |z| active (factor={beta}).")
        else:
            raise ValueError(f"Unknown twas_scaling: {twas_scaling!r}")

        # Apply TWAS scale directly to each receptor's contribution
        df_filled.loc[mask, "weighted_contribution"] = (
            affinity
            * df_filled.loc[mask, "receptor_weight"]
            * df_filled.loc[mask, "twas_scale"]
        )

        # Recompute base_score from the TWAS-scaled per-receptor contributions
        drug_scores = (
            df_filled.groupby("drug", dropna=False)["weighted_contribution"]
            .sum()
            .reset_index()
            .rename(columns={"weighted_contribution": "base_score"})
        )

        # Optional tiny secondary drug-level boost (uses max|z| for stability)
        twas_signal = (
            df_filled.groupby("drug")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
            .reset_index()
            .rename(columns={"twas_z": "max_abs_twas_z"})
        )
        drug_scores = drug_scores.merge(twas_signal, on="drug", how="left")
        drug_scores["max_abs_twas_z"] = drug_scores["max_abs_twas_z"].fillna(0.0)
        boost = 1 + secondary_boost_factor * drug_scores["max_abs_twas_z"]
    else:
        boost = 1.0

    if twas_sig_bonus and "twas_significant" in df.columns:
        sig = df[mask & df["twas_significant"].fillna(False)
                 & (df["receptor_weight"] > 0)]
        n_sig = (
            sig.groupby("drug").size()
            .reset_index(name="n_twas_sig_metabolic")
        )
        drug_scores = drug_scores.merge(n_sig, on="drug", how="left")
        drug_scores["n_twas_sig_metabolic"] = (
            drug_scores["n_twas_sig_metabolic"].fillna(0).astype(int)
        )
        sig_mult = 1 + twas_sig_bonus * drug_scores["n_twas_sig_metabolic"]
    else:
        sig_mult = 1.0

    drug_scores["metabolic_risk_score"] = (
        drug_scores["base_score"] * boost * sig_mult
    )

    if normalize:
        ref_mask = drug_scores["drug"].astype(str).str.lower() \
            == reference_drug.lower()
        ref_score = drug_scores.loc[ref_mask, "metabolic_risk_score"]
        if not ref_score.empty and ref_score.values[0] > 0:
            ref = ref_score.values[0]
        else:
            print(f"  [warn] reference drug '{reference_drug}' not found or "
                  f"zero-scored; normalising to the max score instead.")
            ref = drug_scores["metabolic_risk_score"].max()
        if ref and ref > 0:
            drug_scores["metabolic_risk_score"] = (
                drug_scores["metabolic_risk_score"] / ref * 100
            )
            drug_scores["risk_vs_reference"] = (
                drug_scores["metabolic_risk_score"] - 100
            )

    n_zero_drugs = int((drug_scores["metabolic_risk_score"] == 0).sum())
    if n_zero_drugs:
        drug_scores = drug_scores[drug_scores["metabolic_risk_score"] != 0]
        print(f"  [drop-zero] removed {n_zero_drugs} drug(s) with "
              f"metabolic_risk_score == 0.")

    drug_scores["risk_rank"] = (
        drug_scores["metabolic_risk_score"]
        .rank(ascending=False, method="min")
        .astype("Int64")
    )
    drug_scores = drug_scores.sort_values(
        "metabolic_risk_score", ascending=False
    ).reset_index(drop=True)

    preferred = [
        "risk_rank", "drug", "metabolic_risk_score", "base_score",
        "max_abs_twas_z", "n_twas_sig_metabolic", "risk_vs_reference",
    ]
    cols = [c for c in preferred if c in drug_scores.columns]
    return drug_scores[cols]


# ======================================================================
# Core pipeline
# ======================================================================
def run_pipeline(twas_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    stats = {"twas_dir": twas_dir, "output_dir": output_dir}

    print(f"Building TWAS lookup for: {twas_dir}")
    twas_lookup = build_twas_lookup(twas_dir)
    stats["n_genes_twas"] = len(twas_lookup)

    sample = list(twas_lookup.keys())[:10] if twas_lookup else "EMPTY"
    print(f"  Sample genes in TWAS lookup: {sample}")
    for g in ("HTR2C", "HRH1", "DRD2", "CHRM1", "CHRM3", "HTR2A"):
        print(f"    {g} present? {g in (twas_lookup or {})}")

    print("Loading Ki database ...")
    ki = read_csv_robust(KI_DB_PATH)
    print(f"  Ki rows: {len(ki):,}")

    print("Loading drug list ...")
    drugs = read_csv_robust(DRUGS_CSV_PATH)
    print(f"  Drugs: {len(drugs):,}")
    stats["n_drugs_input"] = len(drugs)

    ligand_col   = find_col(ki, ["Test Ligands", "Test Ligand", "Ligand", "ligand"])
    receptor_col = find_col(ki, ["Receptor", "receptor", "Target"])
    kival_col    = find_col(ki, ["Ki Value", "Ki", "Ki_nM", "Ki (nM)"])
    if not all([ligand_col, receptor_col, kival_col]):
        raise ValueError(
            f"Could not locate required Ki columns. Found: "
            f"ligand={ligand_col}, receptor={receptor_col}, ki={kival_col}\n"
            f"Available columns: {list(ki.columns)}"
        )

    ki["_ligand_norm"] = ki[ligand_col].map(normalise_name)
    ki["_ki_nM"]       = ki[kival_col].map(parse_ki)
    ligand_vocab = sorted(set(ki["_ligand_norm"]) - {""})

    rows = []
    missing_twas = set()

    print("\nMatching drugs and building drug x receptor table ...")
    for _, drug in tqdm(drugs.iterrows(), total=len(drugs)):
        drug_name = drug.get("name")
        norm = normalise_name(drug_name)
        ddd_mg = ddd_to_mg(drug.get("ddd"), drug.get("unit"))
        inv_ddd = (1.0 / ddd_mg) if (not _is_nan(ddd_mg) and ddd_mg > 0) else np.nan

        if not norm:
            continue

        match = process.extractOne(norm, ligand_vocab, scorer=fuzz.WRatio)
        if match is None or match[1] < FUZZY_THRESHOLD:
            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": None,
                "match_score": (match[1] if match else np.nan),
                "receptor": None, "gene": None,
                "Ki_nM": np.nan, "inv_Ki": np.nan,
                "DDD_mg": ddd_mg, "inv_DDD": inv_ddd,
                "affinity_dose_score": np.nan,
                "n_measurements": 0,
                "twas_z": np.nan, "twas_p": np.nan,
                "strong_binder": False, "twas_significant": False,
                "ki_imputed": False, "twas_imputed": False,
            })
            continue

        matched_ligand, score, _ = match
        subset = ki[ki["_ligand_norm"] == matched_ligand]

        grouped = subset.groupby(receptor_col)
        for receptor, grp in grouped:
            ki_nM = aggregate_ki(grp["_ki_nM"], KI_AGGREGATION)
            n_meas = int(grp["_ki_nM"].notna().sum())
            inv_ki = (1.0 / ki_nM) if (not _is_nan(ki_nM) and ki_nM > 0) else np.nan
            gene = receptor_to_gene(receptor)

            if not _is_nan(inv_ki) and not _is_nan(ddd_mg) and ddd_mg > 0:
                score_combined = inv_ki * np.log(ddd_mg + 1.0)
            elif not _is_nan(inv_ki):
                score_combined = inv_ki
            else:
                score_combined = np.nan

            twas = get_twas_for_gene(gene, twas_lookup) if gene else None
            if gene and twas is None:
                missing_twas.add(gene)
            twas_z = twas.get("twas_z", np.nan) if twas else np.nan
            twas_p = twas.get("twas_p", np.nan) if twas else np.nan

            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": matched_ligand,
                "match_score": round(score, 1),
                "receptor": receptor,
                "gene": gene,
                "Ki_nM": ki_nM,
                "inv_Ki": inv_ki,
                "DDD_mg": ddd_mg,
                "inv_DDD": inv_ddd,
                "affinity_dose_score": score_combined,
                "n_measurements": n_meas,
                "twas_z": twas_z,
                "twas_p": twas_p,
                "strong_binder": (not _is_nan(ki_nM)) and ki_nM < STRONG_BINDER_KI_NM,
                "twas_significant": (not _is_nan(twas_p)) and twas_p < TWAS_P_SIGNIFICANT,
                "ki_imputed": False,
                "twas_imputed": False,
            })

    results = pd.DataFrame(rows)

    stats["n_missing_twas_genes"] = len(missing_twas)
    stats["missing_twas_sample"] = sorted(missing_twas)[:12]
    if missing_twas:
        sample = sorted(missing_twas)[:8]
        print(f"\n[TWAS] {len(missing_twas)} receptor gene(s) had no TWAS data "
              f"(e.g. {', '.join(sample)}"
              f"{', ...' if len(missing_twas) > len(sample) else ''})")

    if not results.empty:
        n_before = results["drug"].nunique()
        valid_ddd_drugs = results[results["DDD_mg"].notna()]["drug"].unique()
        results = results[results["drug"].isin(valid_ddd_drugs)].reset_index(drop=True)
        n_after = results["drug"].nunique()
        stats["n_drugs_ddd_dropped"] = int(n_before - n_after)
        print(f"\n[DDD filter] kept {n_after}/{n_before} drug(s) with a valid DDD "
              f"({n_before - n_after} dropped).")

    if not results.empty:
        print("\nImputation step ...")
        results = impute_missing_ki(results, KI_IMPUTE_STRATEGY)
        results = impute_missing_twas(results, TWAS_IMPUTE_STRATEGY)

    results, n_zero_dropped = drop_zero_score_rows(results)
    stats["n_zero_score_dropped"] = n_zero_dropped

    if not results.empty:
        results["rank_in_drug"] = (
            results.groupby("drug")["affinity_dose_score"]
            .rank(ascending=False, method="min")
        )
        results = results.sort_values(
            ["drug", "affinity_dose_score"],
            ascending=[True, False]
        ).reset_index(drop=True)

    if not results.empty:
        stats["n_drugs_matched"] = int(
            results.loc[results["receptor"].notna(), "drug"].nunique()
        )
        stats["n_pairs"] = int(results["receptor"].notna().sum())
        stats["n_ki_imputed"] = int(results.get("ki_imputed",
                                     pd.Series(dtype=bool)).sum())
        stats["n_twas_imputed"] = int(results.get("twas_imputed",
                                       pd.Series(dtype=bool)).sum())
        stats["n_strong_binders"] = int(results["strong_binder"].sum())
        stats["n_twas_significant"] = int(results["twas_significant"].sum())
    else:
        stats.update({
            "n_drugs_matched": 0, "n_pairs": 0, "n_ki_imputed": 0,
            "n_twas_imputed": 0, "n_strong_binders": 0,
            "n_twas_significant": 0,
        })

    return results, stats


def write_outputs(results: pd.DataFrame, output_dir: str):
    paths = {}
    if results.empty:
        print("No results to write.")
        return paths

    csv_path = os.path.join(output_dir, "n05a_ki_twas_full_results.csv")
    results.to_csv(csv_path, index=False)
    print(f"  wrote {csv_path}")
    paths["csv"] = csv_path

    matched = results[results["receptor"].notna()]
    summary = (
        matched.groupby(["atc_code", "drug"])
        .agg(
            n_receptors      = ("receptor", "nunique"),
            n_strong_binders = ("strong_binder", "sum"),
            n_twas_sig       = ("twas_significant", "sum"),
            n_ki_imputed     = ("ki_imputed", "sum"),
            n_twas_imputed   = ("twas_imputed", "sum"),
            best_Ki_nM       = ("Ki_nM", "min"),
            top_score        = ("affinity_dose_score", "max"),
            DDD_mg           = ("DDD_mg", "first"),
        )
        .reset_index()
        .sort_values("top_score", ascending=False)
    )
    summary_path = os.path.join(output_dir, "drug_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"  wrote {summary_path}")
    paths["summary"] = summary_path

    xlsx_path = os.path.join(output_dir, "n05a_ki_twas_full_results.xlsx")
    top5 = (
        matched[matched["rank_in_drug"] <= 5]
        .sort_values(["drug", "rank_in_drug"])
    )
    twas_sig = matched[matched["twas_significant"]].sort_values(
        "affinity_dose_score", ascending=False
    )
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
            results.to_excel(xw, sheet_name="All", index=False)
            top5.to_excel(xw, sheet_name="Top5_per_drug", index=False)
            twas_sig.to_excel(xw, sheet_name="TWAS_significant", index=False)
            summary.to_excel(xw, sheet_name="Drug_summary", index=False)
        print(f"  wrote {xlsx_path}")
        paths["xlsx"] = xlsx_path
    except Exception as e:
        print(f"  (skipped Excel export: {e})")

    return paths


def write_detailed_summary(run_summaries, path: str) -> None:
    L = []
    bar = "=" * 78
    sub = "-" * 78
    L.append(bar)
    L.append("N05A  Ki -> DDD -> TWAS ENRICHMENT PIPELINE — DETAILED SUMMARY")
    L.append("TWAS INTEGRATION: VERSION 3 (Mild exponential scaling on -log10(p))")
    L.append(bar)
    L.append(f"Generated            : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
    L.append(f"Ki database          : {KI_DB_PATH}")
    L.append(f"Drug list            : {DRUGS_CSV_PATH}")
    L.append(f"Output root          : {OUTPUT_DIR}")
    L.append(f"Score formula        : affinity_dose_score = inv_Ki * log(DDD_mg + 1)")
    L.append(f"TWAS scale (v3)      : twas_scale = exp(beta * -log10(p)), "
             f"beta={LOGP_BETA}, cap={MAX_NEGLOG10_P}")
    L.append(f"TWAS dirs processed  : {len(run_summaries)}")
    L.append("")
    L.append("Configuration:")
    L.append(f"  FUZZY_THRESHOLD       = {FUZZY_THRESHOLD}")
    L.append(f"  KI_AGGREGATION        = {KI_AGGREGATION}")
    L.append(f"  STRONG_BINDER_KI_NM   = {STRONG_BINDER_KI_NM}")
    L.append(f"  TWAS_P_SIGNIFICANT    = {TWAS_P_SIGNIFICANT}")
    L.append(f"  KI_IMPUTE_STRATEGY    = {KI_IMPUTE_STRATEGY}")
    L.append(f"  TWAS_IMPUTE_STRATEGY  = {TWAS_IMPUTE_STRATEGY}")
    L.append(f"  LOGP_BETA             = {LOGP_BETA}")
    L.append(f"  MAX_NEGLOG10_P        = {MAX_NEGLOG10_P}")
    L.append("")
    L.append(bar)
    L.append("CROSS-RUN OVERVIEW")
    L.append(bar)
    header = (f"{'TWAS label':<18}{'genes':>8}{'matched':>9}{'pairs':>8}"
              f"{'zero-drop':>11}{'ki_imp':>8}{'tw_imp':>8}{'tw_sig':>8}")
    L.append(header)
    L.append(sub)
    for s in run_summaries:
        if s.get("error"):
            L.append(f"{s.get('label',''):<18}  ERROR: {s['error']}")
            continue
        L.append(
            f"{s.get('label',''):<18}"
            f"{s.get('n_genes_twas',0):>8}"
            f"{s.get('n_drugs_matched',0):>9}"
            f"{s.get('n_pairs',0):>8}"
            f"{s.get('n_zero_score_dropped',0):>11}"
            f"{s.get('n_ki_imputed',0):>8}"
            f"{s.get('n_twas_imputed',0):>8}"
            f"{s.get('n_twas_significant',0):>8}"
        )
    L.append("")
    for s in run_summaries:
        L.append(bar)
        L.append(f"TWAS RUN: {s.get('label','')}")
        L.append(bar)
        L.append(f"  TWAS directory          : {s.get('twas_dir','')}")
        L.append(f"  Output directory        : {s.get('output_dir','')}")
        if s.get("error"):
            L.append(f"  [ERROR] {s['error']}")
            L.append("")
            continue
        L.append(f"  Genes with TWAS data    : {s.get('n_genes_twas',0):,}")
        L.append(f"  Drugs in input list     : {s.get('n_drugs_input',0):,}")
        L.append(f"  Drugs dropped (no DDD)  : {s.get('n_drugs_ddd_dropped',0):,}")
        L.append(f"  Drugs matched to ligand : {s.get('n_drugs_matched',0):,}")
        L.append(f"  Drug x receptor pairs   : {s.get('n_pairs',0):,}")
        L.append(f"  Rows dropped (score==0) : {s.get('n_zero_score_dropped',0):,}")
        L.append(f"  Ki values imputed       : {s.get('n_ki_imputed',0):,}")
        L.append(f"  TWAS values imputed     : {s.get('n_twas_imputed',0):,}")
        L.append(f"  Strong binders          : {s.get('n_strong_binders',0):,}")
        L.append(f"  TWAS-significant rows   : {s.get('n_twas_significant',0):,}")
        L.append(f"  Receptor genes w/o TWAS : {s.get('n_missing_twas_genes',0):,}")
        if s.get("missing_twas_sample"):
            L.append(f"    e.g. {', '.join(s['missing_twas_sample'])}")
        L.append("")
        top_risk = s.get("top_risk")
        if top_risk is not None and len(top_risk):
            L.append("  Top metabolic-risk drugs (reference = 100):")
            L.append(f"    {'rank':>4}  {'drug':<28}{'risk':>10}{'maxTWASz':>10}")
            for _, r in top_risk.iterrows():
                L.append(
                    f"    {str(r.get('risk_rank','')):>4}  "
                    f"{str(r.get('drug',''))[:27]:<28}"
                    f"{r.get('metabolic_risk_score', float('nan')):>10.2f}"
                    f"{r.get('max_abs_twas_z', float('nan')):>10.3f}"
                )
            L.append("")
        top_hits = s.get("top_hits")
        if top_hits is not None and len(top_hits):
            L.append("  Top strong-binding + TWAS-significant hits:")
            L.append(f"    {'drug':<22}{'gene':<9}{'Ki_nM':>10}"
                     f"{'score':>12}{'twas_z':>9}{'twas_p':>11}")
            for _, r in top_hits.iterrows():
                L.append(
                    f"    {str(r.get('drug',''))[:21]:<22}"
                    f"{str(r.get('gene',''))[:8]:<9}"
                    f"{r.get('Ki_nM', float('nan')):>10.3g}"
                    f"{r.get('affinity_dose_score', float('nan')):>12.4g}"
                    f"{r.get('twas_z', float('nan')):>9.3f}"
                    f"{r.get('twas_p', float('nan')):>11.3g}"
                )
            L.append("")
    L.append(bar)
    L.append("END OF SUMMARY")
    L.append(bar)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"\n  wrote detailed summary -> {path}")


# ======================================================================
# MAIN
# ======================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
run_summaries = []

for twas_dir in TWAS_DIRS:
    label = _safe_label(twas_dir)
    out_dir = os.path.join(OUTPUT_DIR, label)
    print("\n" + "#" * 78)
    print(f"# PROCESSING TWAS DIR: {twas_dir}   (label='{label}')")
    print("# TWAS integration = VERSION 3 (mild exponential scaling on -log10(p))")
    print("#" * 78)

    try:
        results, stats = run_pipeline(twas_dir, out_dir)
    except Exception as e:
        print(f"  [ERROR] pipeline failed for {twas_dir}: {e}")
        run_summaries.append({
            "label": label, "twas_dir": twas_dir, "output_dir": out_dir,
            "error": str(e),
        })
        continue

    stats["label"] = label
    n_drugs_matched = stats.get("n_drugs_matched", 0)
    n_pairs = stats.get("n_pairs", 0)
    print(f"\nMatched {n_drugs_matched} drugs to ligands; "
          f"{n_pairs} drug x receptor pairs built.")
    print(f"  imputed Ki rows: {stats.get('n_ki_imputed',0)}  |  "
          f"imputed TWAS rows: {stats.get('n_twas_imputed',0)}")

    print("\nWriting outputs ...")
    write_outputs(results, out_dir)

    print("\nCalculating Metabolic Risk Score (Version 3: logp_expo) ...")
    risk_df = calculate_metabolic_risk_score(
        results,
        weights=METABOLIC_WEIGHTS,
        score_transform="log1p",
        twas_boost=True,
        twas_scaling="logp_expo",           # exp(beta * -log10(p))
        beta=LOGP_BETA,                     # 0.045 (tune 0.03 - 0.06)
        twas_sig_bonus=0.10,
        normalize=True,
        reference_drug="chlorpromazine",
        missing_gene_strategy="mean_abs_z",
        max_neglog10_p=MAX_NEGLOG10_P,
        secondary_boost_factor=SECONDARY_BOOST_FAC,
    )
    risk_path = os.path.join(out_dir, "metabolic_risk_score_per_drug.csv")
    risk_df.to_csv(risk_path, index=False)
    print(f"  wrote {risk_path}")

    print("\nN05A drugs ranked by metabolic risk (reference = 100):")
    print(risk_df.head(20).to_string(index=False))

    stats["top_risk"] = risk_df.head(15).copy()
    preview = results[
        results["twas_significant"] & results["affinity_dose_score"].notna()
    ].sort_values("affinity_dose_score", ascending=False).head(15)
    stats["top_hits"] = preview[
        ["drug", "gene", "Ki_nM", "affinity_dose_score", "twas_z", "twas_p"]
    ].copy() if not preview.empty else None
    if preview.empty:
        print("\n(No TWAS-significant hits found — check TWAS_DIR / column names.)")

    run_summaries.append(stats)
    print(f"\nDone with TWAS dir '{label}'.")

summary_txt = os.path.join(OUTPUT_DIR, "pipeline_summary.txt")
write_detailed_summary(run_summaries, summary_txt)
print("\nAll TWAS directories processed (Version 3 - logp_expo). Done.")


##############################################################################
# PROCESSING TWAS DIR: /content/lipids/ldl   (label='ldl')
# TWAS integration = VERSION 3 (mild exponential scaling on -log10(p))
##############################################################################
Building TWAS lookup for: /content/lipids/ldl
  Pre-loaded TWAS data for 18,057 genes from 6/6 file(s)
  Sample genes in TWAS lookup: ['PCSK9', 'SMARCA4', 'CELSR2', 'ANKDD1B', 'CETP', 'FEN1', 'ATP13A1', 'FADS1', 'SYPL2', 'YIPF2']
    HTR2C present? False
    HRH1 present? True
    DRD2 present? True
    CHRM1 present? True
    CHRM3 present? True
    HTR2A present? True
Loading Ki database ...
  Ki rows: 98,764
Loading drug list ...
  Drugs: 70

Matching drugs and building drug x receptor table ...


100%|██████████| 70/70 [00:02<00:00, 25.21it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.160, twas_p=0.239); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/ldl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/ldl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/ldl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential scalin

100%|██████████| 70/70 [00:03<00:00, 18.66it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.420, twas_p=0.207); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/hdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/hdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/hdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential scalin

...


[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.393, twas_p=0.171); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/logtg/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/logtg/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/logtg/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential 

100%|██████████| 70/70 [00:02<00:00, 24.17it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.089, twas_p=0.327); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/nonhdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/nonhdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/nonhdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponenti

100%|██████████| 70/70 [00:02<00:00, 31.82it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.367, twas_p=0.21); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/tc/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/tc/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/tc/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential scaling on

# Without weights

## Version 4

In [ ]:
# ======================================================================
# VERSION 4:  N05A → Ki → DDD → TWAS pipeline
# TWAS INTEGRATION: MILD EXPONENTIAL SCALING on -log10(p)
#   twas_scale = exp(beta * -log10(p))     (beta ~ 0.03 - 0.06)
# WITHOUT METABOLIC WEIGHTS (USE_METABOLIC_WEIGHTS = False → uniform 1.0)
# Self-contained: run as a single Colab cell.
#   Requires: pip install pandas numpy rapidfuzz tqdm openpyxl
# ======================================================================

import os
import re
import glob
import datetime
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

# ======================================================================
# CONFIG  --  EDIT THESE PATHS
# ======================================================================
KI_DB_PATH     = "/content/Z_database/KiDatabase_2026-06-22.csv"
DRUGS_CSV_PATH = "/content/Z_database/atc_n05a_drugs.csv"

TWAS_DIRS = [
    "/content/lipids/ldl",
    "/content/lipids/hdl",
    "/content/lipids/logtg",
    "/content/lipids/nonhdl",
    "/content/lipids/tc",
]

# Distinct output root for the -log10(p) / no-weights version
OUTPUT_DIR = "/content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights"

# ======================================================================
# METABOLIC WEIGHTS USAGE
# ======================================================================
USE_METABOLIC_WEIGHTS = False          # True = literature weights (original)
                                       # False = uniform weight = 1.0 for all receptors

FUZZY_THRESHOLD = 80
KI_AGGREGATION = "min"
STRONG_BINDER_KI_NM = 10.0
TWAS_P_SIGNIFICANT = 0.05
KI_IMPUTE_STRATEGY = "mean"
TWAS_IMPUTE_STRATEGY = "mean"
CSV_ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

# ---- Version 3/4 (logp_expo) scaling parameters ----
LOGP_BETA           = 0.045   # exponential rate on -log10(p) (tune 0.03 - 0.06)
MAX_NEGLOG10_P      = 25.0    # cap to prevent explosion from ultra-small p-values
SECONDARY_BOOST_FAC = 0.02    # tiny secondary drug-level boost (max|z|)

TWAS_SYMBOL_COL_CANDIDATES = [
    "gene_name", "genename", "gene_symbol", "symbol", "hgnc_symbol",
]
TWAS_ENSG_COL_CANDIDATES = [
    "gene", "ensembl_gene_id", "gene_id", "geneid", "ensembl", "id",
]
TWAS_Z_COL_CANDIDATES = [
    "twas.z", "twas_z", "zscore", "z_score", "zstat", "z", "zscore",
]
TWAS_P_COL_CANDIDATES = [
    "twas.p", "twas_p", "pvalue", "p_value", "pval", "p",
]


# ======================================================================
# Receptor (Ki DB label) -> official HGNC gene symbol
# ======================================================================
RECEPTOR_TO_GENE = {
    "5-HT1A": "HTR1A", "5HT1A": "HTR1A",
    "5-HT1B": "HTR1B", "5HT1B": "HTR1B",
    "5-HT1D": "HTR1D",
    "5-HT1E": "HTR1E",
    "5-HT1F": "HTR1F",
    "5-HT2A": "HTR2A", "5HT2A": "HTR2A",
    "5-HT2B": "HTR2B",
    "5-HT2C": "HTR2C", "5HT2C": "HTR2C",
    "5-HT3":  "HTR3A",
    "5-HT5A": "HTR5A",
    "5-HT6":  "HTR6",
    "5-HT7":  "HTR7",
    "D1": "DRD1", "D2": "DRD2", "D3": "DRD3", "D4": "DRD4", "D5": "DRD5",
    "DRD1": "DRD1", "DRD2": "DRD2", "DRD3": "DRD3", "DRD4": "DRD4", "DRD5": "DRD5",
    "alpha1A": "ADRA1A", "alpha1B": "ADRA1B", "alpha1D": "ADRA1D",
    "alpha2A": "ADRA2A", "alpha2B": "ADRA2B", "alpha2C": "ADRA2C",
    "alpha1":  "ADRA1A", "alpha2": "ADRA2A",
    "beta1": "ADRB1", "beta2": "ADRB2", "beta3": "ADRB3",
    "H1": "HRH1", "H2": "HRH2", "H3": "HRH3", "H4": "HRH4",
    "M1": "CHRM1", "M2": "CHRM2", "M3": "CHRM3", "M4": "CHRM4", "M5": "CHRM5",
    "Muscarinic": "CHRM1",
    "Sigma1": "SIGMAR1", "Sigma 1": "SIGMAR1",
    "SERT": "SLC6A4", "DAT": "SLC6A3", "NET": "SLC6A2",
}
RECEPTOR_TO_GENE = {k.strip().lower(): v for k, v in RECEPTOR_TO_GENE.items()}


# ======================================================================
# METABOLIC / DIABETES RISK WEIGHTS (retained for the comparison run)
# ======================================================================
METABOLIC_WEIGHTS = {
    "HRH1":    1.00,
    "HTR2C":   0.92,
    "CHRM3":   0.85,
    "HTR2A":   0.55,
    "ADRA1A":  0.48,
    "ADRA1B":  0.48,
    "HTR6":    0.42,
    "ADRA2A":  0.32,
    "ADRA2B":  0.30,
    "ADRA2C":  0.30,
    "CHRM1":   0.28,
    "CHRM4":   0.22,
    "CHRM5":   0.20,
    "HTR7":    0.25,
    "DRD3":    0.18,
    "DRD2":    0.12,
    "DRD4":    0.10,
    "HTR1A":   0.08,
    "ADRB1":   0.08,
    "ADRB2":   0.07,
    "SIGMAR1": 0.05,
    "SLC6A4":  0.06,
    "SLC6A2":  0.05,
    "SLC6A3":  0.04,
}


# ======================================================================
# Helpers
# ======================================================================
def _is_nan(x) -> bool:
    if x is None:
        return True
    try:
        return bool(np.isnan(x))
    except (TypeError, ValueError):
        return False


def read_csv_robust(path: str, **kwargs) -> pd.DataFrame:
    kwargs.setdefault("low_memory", False)
    last_err = None
    for enc in CSV_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", **kwargs)
    except Exception:
        raise last_err if last_err else RuntimeError(f"Could not read {path}")


def normalise_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip().lower())


def receptor_to_gene(receptor: str):
    if not isinstance(receptor, str):
        return None
    key = receptor.strip().lower()
    if key in RECEPTOR_TO_GENE:
        return RECEPTOR_TO_GENE[key]
    key2 = re.sub(r"[\s\-]", "", key)
    for k, v in RECEPTOR_TO_GENE.items():
        if re.sub(r"[\s\-]", "", k) == key2:
            return v
    return None


def parse_ki(value) -> float:
    if value is None:
        return np.nan
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"na", "nan", "nd", "n/a", "-"}:
        return np.nan
    s = s.replace(",", "")
    s = re.sub(r"^[><=~]+", "", s)
    m = re.search(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", s)
    return float(m.group()) if m else np.nan


def ddd_to_mg(ddd, unit) -> float:
    ddd_val = parse_ki(ddd)
    if _is_nan(ddd_val):
        return np.nan
    u = (str(unit).strip().lower() if unit is not None else "")
    factors = {"g": 1000.0, "mg": 1.0, "mcg": 0.001, "µg": 0.001, "ug": 0.001}
    factor = factors.get(u, 1.0)
    return ddd_val * factor


def aggregate_ki(values: pd.Series, how: str) -> float:
    vals = values.dropna()
    if vals.empty:
        return np.nan
    if how == "min":
        return float(vals.min())
    if how == "median":
        return float(vals.median())
    if how == "mean":
        return float(vals.mean())
    return float(vals.min())


def find_col(df: pd.DataFrame, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _strip_ensg_version(s) -> str:
    return re.sub(r"\.\d+$", "", str(s).strip().upper())


def _safe_label(path: str) -> str:
    base = os.path.basename(os.path.normpath(path))
    if not base:
        base = re.sub(r"[^A-Za-z0-9_.-]+", "_", path).strip("_")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", base).strip("_") or "twas"


# ======================================================================
# Recompute affinity-derived columns
# ======================================================================
def _recompute_affinity_columns(results: pd.DataFrame) -> pd.DataFrame:
    ki_nM  = results["Ki_nM"]
    ddd_mg = results["DDD_mg"]

    has_ki = ki_nM.notna() & (ki_nM > 0)
    results["inv_Ki"] = np.where(has_ki, 1.0 / ki_nM, np.nan)

    has_inv_ki = results["inv_Ki"].notna()
    has_ddd    = ddd_mg.notna() & (ddd_mg > 0)
    log_ddd    = np.log(ddd_mg.where(has_ddd) + 1.0)
    results["affinity_dose_score"] = np.where(
        has_inv_ki & has_ddd, results["inv_Ki"] * log_ddd,
        np.where(has_inv_ki, results["inv_Ki"], np.nan),
    )

    results["strong_binder"] = ki_nM.notna() & (ki_nM < STRONG_BINDER_KI_NM)
    return results


def drop_zero_score_rows(results: pd.DataFrame):
    if results.empty or "affinity_dose_score" not in results.columns:
        return results, 0
    zero_mask = results["affinity_dose_score"] == 0
    n_dropped = int(zero_mask.sum())
    if n_dropped:
        results = results.loc[~zero_mask].reset_index(drop=True)
        print(f"  [drop-zero] removed {n_dropped} row(s) with affinity_dose_score == 0.")
    else:
        print("  [drop-zero] no rows with affinity_dose_score == 0.")
    return results, n_dropped


def impute_missing_ki(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "ki_imputed" not in results.columns:
        results["ki_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown KI_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid = results.loc[results["Ki_nM"].notna(), ["receptor", "Ki_nM"]]
    if valid.empty:
        print("  [ki-impute] no Ki values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        per_receptor = valid.groupby("receptor")["Ki_nM"].mean()
        global_ref = float(valid["Ki_nM"].mean())
    else:
        per_receptor = valid.groupby("receptor")["Ki_nM"].median()
        global_ref = float(valid["Ki_nM"].median())
    need = results["Ki_nM"].isna() & results["receptor"].notna()
    n_need = int(need.sum())
    if n_need == 0:
        print(f"  [ki-impute] strategy='{strategy}': no missing Ki values to fill.")
        return results
    filled_ref = results.loc[need, "receptor"].map(per_receptor)
    filled_ref = filled_ref.fillna(global_ref)
    results.loc[need, "Ki_nM"] = filled_ref.values
    results.loc[need, "ki_imputed"] = True
    results = _recompute_affinity_columns(results)
    print(f"  [ki-impute] strategy='{strategy}': filled {n_need} missing Ki "
          f"value(s) (global fallback Ki = {global_ref:.3g} nM).")
    return results


def impute_missing_twas(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "twas_imputed" not in results.columns:
        results["twas_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown TWAS_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid_z = results.loc[results["twas_z"].notna(), "twas_z"]
    valid_p = results.loc[results["twas_p"].notna(), "twas_p"]
    if valid_z.empty and valid_p.empty:
        print("  [twas-impute] no TWAS values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        fill_z = float(valid_z.mean()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.mean()) if not valid_p.empty else np.nan
    else:
        fill_z = float(valid_z.median()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.median()) if not valid_p.empty else np.nan
    need_z = results["gene"].notna() & results["twas_z"].isna()
    need_p = results["gene"].notna() & results["twas_p"].isna()
    n_need = int((need_z | need_p).sum())
    if n_need == 0:
        print(f"  [twas-impute] strategy='{strategy}': no missing TWAS values to fill.")
        return results
    if not _is_nan(fill_z):
        results.loc[need_z, "twas_z"] = fill_z
    if not _is_nan(fill_p):
        results.loc[need_p, "twas_p"] = fill_p
    results.loc[need_z | need_p, "twas_imputed"] = True
    print(f"  [twas-impute] strategy='{strategy}': filled {n_need} gene row(s) "
          f"(twas_z={fill_z:.3f}, twas_p={fill_p:.3g}); "
          f"imputed rows kept NON-significant.")
    return results


# ======================================================================
# TWAS lookup
# ======================================================================
def _read_twas_file(fpath: str):
    for sep in [",", "\t", None]:
        for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
            try:
                df = pd.read_csv(
                    fpath, sep=sep, encoding=enc, engine="python",
                    low_memory=False, on_bad_lines="skip"
                )
                if df is not None and not df.empty and df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    try:
        df = pd.read_csv(fpath, engine="python", on_bad_lines="skip")
        if df is not None and not df.empty:
            return df
    except Exception:
        pass
    return None


def build_twas_lookup(twas_dir: str):
    lookup = {}
    files = glob.glob(os.path.join(twas_dir, "*.*"))
    if not files:
        print(f"  [warn] No files found in {twas_dir}")
        return lookup
    successful_files = 0
    skipped_files = []
    for fpath in files:
        try:
            df = _read_twas_file(fpath)
            if df is None or df.empty:
                skipped_files.append((os.path.basename(fpath), "unreadable/empty"))
                continue
            sym_col  = find_col(df, TWAS_SYMBOL_COL_CANDIDATES)
            ensg_col = find_col(df, TWAS_ENSG_COL_CANDIDATES)
            z_col    = find_col(df, TWAS_Z_COL_CANDIDATES)
            p_col    = find_col(df, TWAS_P_COL_CANDIDATES)
            if ensg_col is not None and ensg_col == sym_col:
                ensg_col = None
            if z_col is None or (sym_col is None and ensg_col is None):
                skipped_files.append((
                    os.path.basename(fpath),
                    f"missing cols (z={z_col}, sym={sym_col}, ensg={ensg_col}); "
                    f"had {list(df.columns)[:8]}"
                ))
                continue
            n = len(df)
            sym_vals  = (df[sym_col].astype(str).str.strip().str.upper()
                         if sym_col else pd.Series([""] * n, index=df.index))
            ensg_vals = (df[ensg_col].map(_strip_ensg_version)
                         if ensg_col else pd.Series([""] * n, index=df.index))
            z_vals = df[z_col].map(parse_ki)
            p_vals = (df[p_col].map(parse_ki)
                      if p_col else pd.Series([np.nan] * n, index=df.index))
            bad = {"NAN", "NA", "NONE", ""}
            for i in df.index:
                gene = sym_vals[i]
                if gene in bad:
                    gene = ensg_vals[i]
                if gene in bad:
                    continue
                z = z_vals[i]
                p = p_vals[i]
                if _is_nan(z) and _is_nan(p):
                    continue
                prev = lookup.get(gene)
                if prev is None:
                    lookup[gene] = {"twas_z": z, "twas_p": p}
                else:
                    prev_p = prev.get("twas_p", np.nan)
                    if not _is_nan(p) and (_is_nan(prev_p) or p < prev_p):
                        lookup[gene] = {"twas_z": z, "twas_p": p}
            successful_files += 1
        except Exception as e:
            skipped_files.append((os.path.basename(fpath), f"error: {e}"))
            continue
    print(f"  Pre-loaded TWAS data for {len(lookup):,} genes "
          f"from {successful_files}/{len(files)} file(s)")
    if skipped_files:
        print(f"  [warn] {len(skipped_files)} file(s) skipped:")
        for name, reason in skipped_files[:8]:
            print(f"         - {name}: {reason}")
    return lookup


def get_twas_for_gene(gene: str, lookup):
    if gene is None or lookup is None:
        return None
    return lookup.get(str(gene).strip().upper())


# ======================================================================
# Metabolic risk score  --  VERSION 4: EXPONENTIAL scaling on -log10(p)
#                          supports uniform (no) weights
# ======================================================================
def calculate_metabolic_risk_score(
    df: pd.DataFrame,
    weights: dict = None,
    use_metabolic_weights: bool = True,      # NEW
    score_transform: str = "log1p",
    twas_boost: bool = True,
    twas_scaling: str = "logp_expo",         # "logp_expo" | "logp_linear" | "z_expo" | "z_linear" | "none"
    beta: float = 0.045,
    gamma: float = 0.12,
    twas_sig_bonus: float = 0.10,
    normalize: bool = True,
    reference_drug: str = "chlorpromazine",
    missing_gene_strategy: str = "mean_abs_z",
    max_neglog10_p: float = 25.0,
    secondary_boost_factor: float = 0.02,
) -> pd.DataFrame:
    """
    Per-drug risk score for antipsychotics.

    When use_metabolic_weights=False, every recognised receptor gene gets
    weight = 1.0 (uniform, non-circular). Per-receptor TWAS scaling
    (logp_expo by default) is applied across ALL genes with TWAS data.
    """
    if weights is None:
        weights = METABOLIC_WEIGHTS

    df = df.copy()
    mask = df["gene"].notna() & df["affinity_dose_score"].notna()

    # === Determine which genes get differential weighting ===
    if use_metabolic_weights:
        active_weights = weights
        metab_mask = mask & df["gene"].isin(active_weights.keys())
        print("  [weights] Using literature-based METABOLIC_WEIGHTS")
    else:
        active_weights = {g: 1.0 for g in df.loc[mask, "gene"].dropna().unique()}
        metab_mask = mask & df["gene"].notna()
        print("  [weights] METABOLIC WEIGHTS DISABLED → uniform weight = 1.0")

    # ---- report TWAS coverage among scored genes ----
    n_metab = int(metab_mask.sum())
    uses_p = twas_scaling in ("logp_expo", "logp_linear")
    cov_col = "twas_p" if uses_p else "twas_z"
    n_metab_twas = int((metab_mask & df[cov_col].notna()).sum())
    if twas_boost and n_metab > 0:
        cov = n_metab_twas / n_metab
        if cov < 0.5:
            print(f"  [note] TWAS coverage is low: only {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) scored-gene rows have a TWAS {cov_col}. "
                  f"Per-receptor scaling will be partial.")
        else:
            print(f"  [note] TWAS coverage: {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) scored-gene rows have a TWAS {cov_col}.")

    if score_transform == "log1p":
        affinity = np.log1p(df.loc[mask, "affinity_dose_score"].clip(lower=0))
    elif score_transform == "none":
        affinity = df.loc[mask, "affinity_dose_score"]
    else:
        raise ValueError(f"Unknown score_transform: {score_transform!r}")

    df.loc[mask, "receptor_weight"] = (
        df.loc[mask, "gene"].map(active_weights).fillna(0.0)
    )
    df.loc[mask, "weighted_contribution"] = affinity * df.loc[mask, "receptor_weight"]

    drug_scores = (
        df.groupby("drug", dropna=False)["weighted_contribution"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_contribution": "base_score"})
    )

    # ==================================================================
    # TWAS integration - Version 4: exponential on -log10(p)
    # ==================================================================
    if twas_boost and twas_scaling != "none":
        df_filled = df.copy()

        # --- Fairness imputation for missing scored genes ---
        if missing_gene_strategy == "mean_abs_z":
            if uses_p:
                avail_p = df_filled.loc[
                    metab_mask & df_filled["twas_p"].notna(), "twas_p"
                ].clip(lower=1e-300)
                if len(avail_p) > 0:
                    mean_neglogp = float((-np.log10(avail_p)).mean())
                else:
                    mean_neglogp = 0.0
                impute_mask = (
                    mask
                    & df_filled["twas_p"].isna()
                    & df_filled["gene"].isin(active_weights.keys())
                )
                n_imputed = int(impute_mask.sum())
                if n_imputed > 0:
                    df_filled.loc[impute_mask, "twas_p"] = 10.0 ** (-mean_neglogp)
                    print(f"  [fairness] imputed {n_imputed} missing scored "
                          f"p-value(s) with mean -log10(p) = {mean_neglogp:.3f}")
            else:
                available_z = df_filled.loc[
                    metab_mask & df_filled["twas_z"].notna(), "twas_z"
                ].abs()
                mean_abs_z = float(available_z.mean()) if len(available_z) > 0 else 0.0
                impute_mask = (
                    mask
                    & df_filled["twas_z"].isna()
                    & df_filled["gene"].isin(active_weights.keys())
                )
                n_imputed = int(impute_mask.sum())
                if n_imputed > 0:
                    df_filled.loc[impute_mask, "twas_z"] = mean_abs_z
                    print(f"  [fairness] imputed {n_imputed} missing scored-gene "
                          f"row(s) with mean |twas_z| = {mean_abs_z:.3f}")
        elif missing_gene_strategy == "zero":
            print("  [fairness] missing_gene_strategy='zero' "
                  "(no TWAS boost for missing genes)")
        else:
            raise ValueError(
                f"Unknown missing_gene_strategy: {missing_gene_strategy!r} "
                f"(expected 'mean_abs_z' or 'zero')"
            )

        # --- Per-receptor TWAS scaling ---
        if twas_scaling == "logp_expo":
            p_safe = df_filled.loc[mask, "twas_p"].clip(lower=1e-300)
            neglog10_p = (-np.log10(p_safe)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = np.exp(beta * neglog10_p)
            print(f"  [TWAS-logp] exponential scaling on -log10(p) active "
                  f"(beta={beta}, cap={max_neglog10_p}). "
                  f"Secondary drug-level boost factor = {secondary_boost_factor}")
        elif twas_scaling == "logp_linear":
            p_safe = df_filled.loc[mask, "twas_p"].clip(lower=1e-300)
            neglog10_p = (-np.log10(p_safe)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = 1.0 + gamma * neglog10_p
            print(f"  [TWAS-logp] linear scaling on -log10(p) active "
                  f"(gamma={gamma}, cap={max_neglog10_p}).")
        elif twas_scaling == "z_expo":
            twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = np.exp(beta * twas_abs)
            print(f"  [TWAS-z] exponential scaling on |z| active (beta={beta}).")
        elif twas_scaling == "z_linear":
            twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = 1.0 + beta * twas_abs
            print(f"  [TWAS-z] linear scaling on |z| active (factor={beta}).")
        else:
            raise ValueError(f"Unknown twas_scaling: {twas_scaling!r}")

        df_filled.loc[mask, "weighted_contribution"] = (
            affinity
            * df_filled.loc[mask, "receptor_weight"]
            * df_filled.loc[mask, "twas_scale"]
        )

        drug_scores = (
            df_filled.groupby("drug", dropna=False)["weighted_contribution"]
            .sum()
            .reset_index()
            .rename(columns={"weighted_contribution": "base_score"})
        )

        twas_signal = (
            df_filled.groupby("drug")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
            .reset_index()
            .rename(columns={"twas_z": "max_abs_twas_z"})
        )
        drug_scores = drug_scores.merge(twas_signal, on="drug", how="left")
        drug_scores["max_abs_twas_z"] = drug_scores["max_abs_twas_z"].fillna(0.0)
        boost = 1 + secondary_boost_factor * drug_scores["max_abs_twas_z"]
    else:
        boost = 1.0

    if twas_sig_bonus and "twas_significant" in df.columns:
        sig = df[mask & df["twas_significant"].fillna(False)
                 & (df["receptor_weight"] > 0)]
        n_sig = (
            sig.groupby("drug").size()
            .reset_index(name="n_twas_sig_metabolic")
        )
        drug_scores = drug_scores.merge(n_sig, on="drug", how="left")
        drug_scores["n_twas_sig_metabolic"] = (
            drug_scores["n_twas_sig_metabolic"].fillna(0).astype(int)
        )
        sig_mult = 1 + twas_sig_bonus * drug_scores["n_twas_sig_metabolic"]
    else:
        sig_mult = 1.0

    drug_scores["metabolic_risk_score"] = (
        drug_scores["base_score"] * boost * sig_mult
    )

    if normalize:
        ref_mask = drug_scores["drug"].astype(str).str.lower() \
            == reference_drug.lower()
        ref_score = drug_scores.loc[ref_mask, "metabolic_risk_score"]
        if not ref_score.empty and ref_score.values[0] > 0:
            ref = ref_score.values[0]
        else:
            print(f"  [warn] reference drug '{reference_drug}' not found or "
                  f"zero-scored; normalising to the max score instead.")
            ref = drug_scores["metabolic_risk_score"].max()
        if ref and ref > 0:
            drug_scores["metabolic_risk_score"] = (
                drug_scores["metabolic_risk_score"] / ref * 100
            )
            drug_scores["risk_vs_reference"] = (
                drug_scores["metabolic_risk_score"] - 100
            )

    n_zero_drugs = int((drug_scores["metabolic_risk_score"] == 0).sum())
    if n_zero_drugs:
        drug_scores = drug_scores[drug_scores["metabolic_risk_score"] != 0]
        print(f"  [drop-zero] removed {n_zero_drugs} drug(s) with "
              f"metabolic_risk_score == 0.")

    drug_scores["risk_rank"] = (
        drug_scores["metabolic_risk_score"]
        .rank(ascending=False, method="min")
        .astype("Int64")
    )
    drug_scores = drug_scores.sort_values(
        "metabolic_risk_score", ascending=False
    ).reset_index(drop=True)

    preferred = [
        "risk_rank", "drug", "metabolic_risk_score", "base_score",
        "max_abs_twas_z", "n_twas_sig_metabolic", "risk_vs_reference",
    ]
    cols = [c for c in preferred if c in drug_scores.columns]
    return drug_scores[cols]


# ======================================================================
# Core pipeline
# ======================================================================
def run_pipeline(twas_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    stats = {"twas_dir": twas_dir, "output_dir": output_dir}

    print(f"Building TWAS lookup for: {twas_dir}")
    twas_lookup = build_twas_lookup(twas_dir)
    stats["n_genes_twas"] = len(twas_lookup)

    sample = list(twas_lookup.keys())[:10] if twas_lookup else "EMPTY"
    print(f"  Sample genes in TWAS lookup: {sample}")
    for g in ("HTR2C", "HRH1", "DRD2", "CHRM1", "CHRM3", "HTR2A"):
        print(f"    {g} present? {g in (twas_lookup or {})}")

    print("Loading Ki database ...")
    ki = read_csv_robust(KI_DB_PATH)
    print(f"  Ki rows: {len(ki):,}")

    print("Loading drug list ...")
    drugs = read_csv_robust(DRUGS_CSV_PATH)
    print(f"  Drugs: {len(drugs):,}")
    stats["n_drugs_input"] = len(drugs)

    ligand_col   = find_col(ki, ["Test Ligands", "Test Ligand", "Ligand", "ligand"])
    receptor_col = find_col(ki, ["Receptor", "receptor", "Target"])
    kival_col    = find_col(ki, ["Ki Value", "Ki", "Ki_nM", "Ki (nM)"])
    if not all([ligand_col, receptor_col, kival_col]):
        raise ValueError(
            f"Could not locate required Ki columns. Found: "
            f"ligand={ligand_col}, receptor={receptor_col}, ki={kival_col}\n"
            f"Available columns: {list(ki.columns)}"
        )

    ki["_ligand_norm"] = ki[ligand_col].map(normalise_name)
    ki["_ki_nM"]       = ki[kival_col].map(parse_ki)
    ligand_vocab = sorted(set(ki["_ligand_norm"]) - {""})

    rows = []
    missing_twas = set()

    print("\nMatching drugs and building drug x receptor table ...")
    for _, drug in tqdm(drugs.iterrows(), total=len(drugs)):
        drug_name = drug.get("name")
        norm = normalise_name(drug_name)
        ddd_mg = ddd_to_mg(drug.get("ddd"), drug.get("unit"))
        inv_ddd = (1.0 / ddd_mg) if (not _is_nan(ddd_mg) and ddd_mg > 0) else np.nan

        if not norm:
            continue

        match = process.extractOne(norm, ligand_vocab, scorer=fuzz.WRatio)
        if match is None or match[1] < FUZZY_THRESHOLD:
            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": None,
                "match_score": (match[1] if match else np.nan),
                "receptor": None, "gene": None,
                "Ki_nM": np.nan, "inv_Ki": np.nan,
                "DDD_mg": ddd_mg, "inv_DDD": inv_ddd,
                "affinity_dose_score": np.nan,
                "n_measurements": 0,
                "twas_z": np.nan, "twas_p": np.nan,
                "strong_binder": False, "twas_significant": False,
                "ki_imputed": False, "twas_imputed": False,
            })
            continue

        matched_ligand, score, _ = match
        subset = ki[ki["_ligand_norm"] == matched_ligand]

        grouped = subset.groupby(receptor_col)
        for receptor, grp in grouped:
            ki_nM = aggregate_ki(grp["_ki_nM"], KI_AGGREGATION)
            n_meas = int(grp["_ki_nM"].notna().sum())
            inv_ki = (1.0 / ki_nM) if (not _is_nan(ki_nM) and ki_nM > 0) else np.nan
            gene = receptor_to_gene(receptor)

            if not _is_nan(inv_ki) and not _is_nan(ddd_mg) and ddd_mg > 0:
                score_combined = inv_ki * np.log(ddd_mg + 1.0)
            elif not _is_nan(inv_ki):
                score_combined = inv_ki
            else:
                score_combined = np.nan

            twas = get_twas_for_gene(gene, twas_lookup) if gene else None
            if gene and twas is None:
                missing_twas.add(gene)
            twas_z = twas.get("twas_z", np.nan) if twas else np.nan
            twas_p = twas.get("twas_p", np.nan) if twas else np.nan

            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": matched_ligand,
                "match_score": round(score, 1),
                "receptor": receptor,
                "gene": gene,
                "Ki_nM": ki_nM,
                "inv_Ki": inv_ki,
                "DDD_mg": ddd_mg,
                "inv_DDD": inv_ddd,
                "affinity_dose_score": score_combined,
                "n_measurements": n_meas,
                "twas_z": twas_z,
                "twas_p": twas_p,
                "strong_binder": (not _is_nan(ki_nM)) and ki_nM < STRONG_BINDER_KI_NM,
                "twas_significant": (not _is_nan(twas_p)) and twas_p < TWAS_P_SIGNIFICANT,
                "ki_imputed": False,
                "twas_imputed": False,
            })

    results = pd.DataFrame(rows)

    stats["n_missing_twas_genes"] = len(missing_twas)
    stats["missing_twas_sample"] = sorted(missing_twas)[:12]
    if missing_twas:
        sample = sorted(missing_twas)[:8]
        print(f"\n[TWAS] {len(missing_twas)} receptor gene(s) had no TWAS data "
              f"(e.g. {', '.join(sample)}"
              f"{', ...' if len(missing_twas) > len(sample) else ''})")

    if not results.empty:
        n_before = results["drug"].nunique()
        valid_ddd_drugs = results[results["DDD_mg"].notna()]["drug"].unique()
        results = results[results["drug"].isin(valid_ddd_drugs)].reset_index(drop=True)
        n_after = results["drug"].nunique()
        stats["n_drugs_ddd_dropped"] = int(n_before - n_after)
        print(f"\n[DDD filter] kept {n_after}/{n_before} drug(s) with a valid DDD "
              f"({n_before - n_after} dropped).")

    if not results.empty:
        print("\nImputation step ...")
        results = impute_missing_ki(results, KI_IMPUTE_STRATEGY)
        results = impute_missing_twas(results, TWAS_IMPUTE_STRATEGY)

    results, n_zero_dropped = drop_zero_score_rows(results)
    stats["n_zero_score_dropped"] = n_zero_dropped

    if not results.empty:
        results["rank_in_drug"] = (
            results.groupby("drug")["affinity_dose_score"]
            .rank(ascending=False, method="min")
        )
        results = results.sort_values(
            ["drug", "affinity_dose_score"],
            ascending=[True, False]
        ).reset_index(drop=True)

    if not results.empty:
        stats["n_drugs_matched"] = int(
            results.loc[results["receptor"].notna(), "drug"].nunique()
        )
        stats["n_pairs"] = int(results["receptor"].notna().sum())
        stats["n_ki_imputed"] = int(results.get("ki_imputed",
                                     pd.Series(dtype=bool)).sum())
        stats["n_twas_imputed"] = int(results.get("twas_imputed",
                                       pd.Series(dtype=bool)).sum())
        stats["n_strong_binders"] = int(results["strong_binder"].sum())
        stats["n_twas_significant"] = int(results["twas_significant"].sum())
    else:
        stats.update({
            "n_drugs_matched": 0, "n_pairs": 0, "n_ki_imputed": 0,
            "n_twas_imputed": 0, "n_strong_binders": 0,
            "n_twas_significant": 0,
        })

    return results, stats


def write_outputs(results: pd.DataFrame, output_dir: str):
    paths = {}
    if results.empty:
        print("No results to write.")
        return paths

    csv_path = os.path.join(output_dir, "n05a_ki_twas_full_results.csv")
    results.to_csv(csv_path, index=False)
    print(f"  wrote {csv_path}")
    paths["csv"] = csv_path

    matched = results[results["receptor"].notna()]
    summary = (
        matched.groupby(["atc_code", "drug"])
        .agg(
            n_receptors      = ("receptor", "nunique"),
            n_strong_binders = ("strong_binder", "sum"),
            n_twas_sig       = ("twas_significant", "sum"),
            n_ki_imputed     = ("ki_imputed", "sum"),
            n_twas_imputed   = ("twas_imputed", "sum"),
            best_Ki_nM       = ("Ki_nM", "min"),
            top_score        = ("affinity_dose_score", "max"),
            DDD_mg           = ("DDD_mg", "first"),
        )
        .reset_index()
        .sort_values("top_score", ascending=False)
    )
    summary_path = os.path.join(output_dir, "drug_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"  wrote {summary_path}")
    paths["summary"] = summary_path

    xlsx_path = os.path.join(output_dir, "n05a_ki_twas_full_results.xlsx")
    top5 = (
        matched[matched["rank_in_drug"] <= 5]
        .sort_values(["drug", "rank_in_drug"])
    )
    twas_sig = matched[matched["twas_significant"]].sort_values(
        "affinity_dose_score", ascending=False
    )
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
            results.to_excel(xw, sheet_name="All", index=False)
            top5.to_excel(xw, sheet_name="Top5_per_drug", index=False)
            twas_sig.to_excel(xw, sheet_name="TWAS_significant", index=False)
            summary.to_excel(xw, sheet_name="Drug_summary", index=False)
        print(f"  wrote {xlsx_path}")
        paths["xlsx"] = xlsx_path
    except Exception as e:
        print(f"  (skipped Excel export: {e})")

    return paths


def write_detailed_summary(run_summaries, path: str) -> None:
    L = []
    bar = "=" * 78
    sub = "-" * 78
    L.append(bar)
    L.append("N05A  Ki -> DDD -> TWAS ENRICHMENT PIPELINE — DETAILED SUMMARY")
    L.append("TWAS INTEGRATION: VERSION 4 (Mild exponential scaling on -log10(p))")
    L.append("WEIGHTING MODE: "
             + ("LITERATURE METABOLIC_WEIGHTS" if USE_METABOLIC_WEIGHTS
                else "UNIFORM WEIGHTS = 1.0 (no metabolic weights)"))
    L.append(bar)
    L.append(f"Generated            : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
    L.append(f"Ki database          : {KI_DB_PATH}")
    L.append(f"Drug list            : {DRUGS_CSV_PATH}")
    L.append(f"Output root          : {OUTPUT_DIR}")
    L.append(f"Score formula        : affinity_dose_score = inv_Ki * log(DDD_mg + 1)")
    L.append(f"TWAS scale (v4)      : twas_scale = exp(beta * -log10(p)), "
             f"beta={LOGP_BETA}, cap={MAX_NEGLOG10_P}")
    L.append(f"USE_METABOLIC_WEIGHTS: {USE_METABOLIC_WEIGHTS}")
    L.append(f"TWAS dirs processed  : {len(run_summaries)}")
    L.append("")
    L.append("Configuration:")
    L.append(f"  FUZZY_THRESHOLD       = {FUZZY_THRESHOLD}")
    L.append(f"  KI_AGGREGATION        = {KI_AGGREGATION}")
    L.append(f"  STRONG_BINDER_KI_NM   = {STRONG_BINDER_KI_NM}")
    L.append(f"  TWAS_P_SIGNIFICANT    = {TWAS_P_SIGNIFICANT}")
    L.append(f"  KI_IMPUTE_STRATEGY    = {KI_IMPUTE_STRATEGY}")
    L.append(f"  TWAS_IMPUTE_STRATEGY  = {TWAS_IMPUTE_STRATEGY}")
    L.append(f"  LOGP_BETA             = {LOGP_BETA}")
    L.append(f"  MAX_NEGLOG10_P        = {MAX_NEGLOG10_P}")
    L.append("")
    L.append(bar)
    L.append("CROSS-RUN OVERVIEW")
    L.append(bar)
    header = (f"{'TWAS label':<18}{'genes':>8}{'matched':>9}{'pairs':>8}"
              f"{'zero-drop':>11}{'ki_imp':>8}{'tw_imp':>8}{'tw_sig':>8}")
    L.append(header)
    L.append(sub)
    for s in run_summaries:
        if s.get("error"):
            L.append(f"{s.get('label',''):<18}  ERROR: {s['error']}")
            continue
        L.append(
            f"{s.get('label',''):<18}"
            f"{s.get('n_genes_twas',0):>8}"
            f"{s.get('n_drugs_matched',0):>9}"
            f"{s.get('n_pairs',0):>8}"
            f"{s.get('n_zero_score_dropped',0):>11}"
            f"{s.get('n_ki_imputed',0):>8}"
            f"{s.get('n_twas_imputed',0):>8}"
            f"{s.get('n_twas_significant',0):>8}"
        )
    L.append("")
    for s in run_summaries:
        L.append(bar)
        L.append(f"TWAS RUN: {s.get('label','')}")
        L.append(bar)
        L.append(f"  TWAS directory          : {s.get('twas_dir','')}")
        L.append(f"  Output directory        : {s.get('output_dir','')}")
        if s.get("error"):
            L.append(f"  [ERROR] {s['error']}")
            L.append("")
            continue
        L.append(f"  Genes with TWAS data    : {s.get('n_genes_twas',0):,}")
        L.append(f"  Drugs in input list     : {s.get('n_drugs_input',0):,}")
        L.append(f"  Drugs dropped (no DDD)  : {s.get('n_drugs_ddd_dropped',0):,}")
        L.append(f"  Drugs matched to ligand : {s.get('n_drugs_matched',0):,}")
        L.append(f"  Drug x receptor pairs   : {s.get('n_pairs',0):,}")
        L.append(f"  Rows dropped (score==0) : {s.get('n_zero_score_dropped',0):,}")
        L.append(f"  Ki values imputed       : {s.get('n_ki_imputed',0):,}")
        L.append(f"  TWAS values imputed     : {s.get('n_twas_imputed',0):,}")
        L.append(f"  Strong binders          : {s.get('n_strong_binders',0):,}")
        L.append(f"  TWAS-significant rows   : {s.get('n_twas_significant',0):,}")
        L.append(f"  Receptor genes w/o TWAS : {s.get('n_missing_twas_genes',0):,}")
        if s.get("missing_twas_sample"):
            L.append(f"    e.g. {', '.join(s['missing_twas_sample'])}")
        L.append("")
        top_risk = s.get("top_risk")
        if top_risk is not None and len(top_risk):
            L.append("  Top risk drugs (reference = 100):")
            L.append(f"    {'rank':>4}  {'drug':<28}{'risk':>10}{'maxTWASz':>10}")
            for _, r in top_risk.iterrows():
                L.append(
                    f"    {str(r.get('risk_rank','')):>4}  "
                    f"{str(r.get('drug',''))[:27]:<28}"
                    f"{r.get('metabolic_risk_score', float('nan')):>10.2f}"
                    f"{r.get('max_abs_twas_z', float('nan')):>10.3f}"
                )
            L.append("")
        top_hits = s.get("top_hits")
        if top_hits is not None and len(top_hits):
            L.append("  Top strong-binding + TWAS-significant hits:")
            L.append(f"    {'drug':<22}{'gene':<9}{'Ki_nM':>10}"
                     f"{'score':>12}{'twas_z':>9}{'twas_p':>11}")
            for _, r in top_hits.iterrows():
                L.append(
                    f"    {str(r.get('drug',''))[:21]:<22}"
                    f"{str(r.get('gene',''))[:8]:<9}"
                    f"{r.get('Ki_nM', float('nan')):>10.3g}"
                    f"{r.get('affinity_dose_score', float('nan')):>12.4g}"
                    f"{r.get('twas_z', float('nan')):>9.3f}"
                    f"{r.get('twas_p', float('nan')):>11.3g}"
                )
            L.append("")
    L.append(bar)
    L.append("END OF SUMMARY")
    L.append(bar)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"\n  wrote detailed summary -> {path}")


# ======================================================================
# MAIN
# ======================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
run_summaries = []

# filename suffix reflecting the weighting mode
suffix = "with_metabolic_weights" if USE_METABOLIC_WEIGHTS else "uniform_weights"

for twas_dir in TWAS_DIRS:
    label = _safe_label(twas_dir)
    out_dir = os.path.join(OUTPUT_DIR, label)
    print("\n" + "#" * 78)
    print(f"# PROCESSING TWAS DIR: {twas_dir}   (label='{label}')")
    print("# TWAS integration = VERSION 4 (mild exponential scaling on -log10(p))")
    print(f"# WEIGHTING MODE: USE_METABOLIC_WEIGHTS={USE_METABOLIC_WEIGHTS}")
    print("#" * 78)

    try:
        results, stats = run_pipeline(twas_dir, out_dir)
    except Exception as e:
        print(f"  [ERROR] pipeline failed for {twas_dir}: {e}")
        run_summaries.append({
            "label": label, "twas_dir": twas_dir, "output_dir": out_dir,
            "error": str(e),
        })
        continue

    stats["label"] = label
    n_drugs_matched = stats.get("n_drugs_matched", 0)
    n_pairs = stats.get("n_pairs", 0)
    print(f"\nMatched {n_drugs_matched} drugs to ligands; "
          f"{n_pairs} drug x receptor pairs built.")
    print(f"  imputed Ki rows: {stats.get('n_ki_imputed',0)}  |  "
          f"imputed TWAS rows: {stats.get('n_twas_imputed',0)}")

    print("\nWriting outputs ...")
    write_outputs(results, out_dir)

    print("\nCalculating Risk Score (Version 4: logp_expo) ...")
    risk_df = calculate_metabolic_risk_score(
        results,
        weights=METABOLIC_WEIGHTS,
        use_metabolic_weights=USE_METABOLIC_WEIGHTS,   # ← added
        score_transform="log1p",
        twas_boost=True,
        twas_scaling="logp_expo",           # exp(beta * -log10(p))
        beta=LOGP_BETA,                     # 0.045 (tune 0.03 - 0.06)
        twas_sig_bonus=0.10,
        normalize=True,
        reference_drug="chlorpromazine",
        missing_gene_strategy="mean_abs_z",
        max_neglog10_p=MAX_NEGLOG10_P,
        secondary_boost_factor=SECONDARY_BOOST_FAC,
    )
    risk_path = os.path.join(
        out_dir, f"metabolic_risk_score_per_drug_{suffix}.csv"
    )
    risk_df.to_csv(risk_path, index=False)
    print(f"  wrote {risk_path}")

    print("\nN05A drugs ranked by risk (reference = 100):")
    print(risk_df.head(20).to_string(index=False))

    stats["top_risk"] = risk_df.head(15).copy()
    preview = results[
        results["twas_significant"] & results["affinity_dose_score"].notna()
    ].sort_values("affinity_dose_score", ascending=False).head(15)
    stats["top_hits"] = preview[
        ["drug", "gene", "Ki_nM", "affinity_dose_score", "twas_z", "twas_p"]
    ].copy() if not preview.empty else None
    if preview.empty:
        print("\n(No TWAS-significant hits found — check TWAS_DIR / column names.)")

    run_summaries.append(stats)
    print(f"\nDone with TWAS dir '{label}'.")

summary_txt = os.path.join(OUTPUT_DIR, f"pipeline_summary_{suffix}.txt")
write_detailed_summary(run_summaries, summary_txt)
print("\nAll TWAS directories processed (Version 4 - logp_expo, no weights). Done.")


##############################################################################
# PROCESSING TWAS DIR: /content/lipids/ldl   (label='ldl')
# TWAS integration = VERSION 4 (mild exponential scaling on -log10(p))
# WEIGHTING MODE: USE_METABOLIC_WEIGHTS=False
##############################################################################
Building TWAS lookup for: /content/lipids/ldl
  Pre-loaded TWAS data for 18,057 genes from 6/6 file(s)
  Sample genes in TWAS lookup: ['PCSK9', 'SMARCA4', 'CELSR2', 'ANKDD1B', 'CETP', 'FEN1', 'ATP13A1', 'FADS1', 'SYPL2', 'YIPF2']
    HTR2C present? False
    HRH1 present? True
    DRD2 present? True
    CHRM1 present? True
    CHRM3 present? True
    HTR2A present? True
Loading Ki database ...
  Ki rows: 98,764
Loading drug list ...
  Drugs: 70

Matching drugs and building drug x receptor table ...


100%|██████████| 70/70 [00:02<00:00, 31.69it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.160, twas_p=0.239); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/ldl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/ldl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/ldl/n05a_ki_twas_full_results.xlsx

Calculating Risk Score (Version 4: logp_expo) ...
  [weights] METABOLIC WEIGHTS DISABLED → uniform weight = 1.0
  [note] TWAS coverage: 960/960 (

100%|██████████| 70/70 [00:02<00:00, 32.06it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.420, twas_p=0.207); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/hdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/hdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/hdl/n05a_ki_twas_full_results.xlsx

Calculating Risk Score (Version 4: logp_expo) ...
  [weights] METABOLIC WEIGHTS DISABLED → uniform weight = 1.0
  [note] TWAS coverage: 960/960 (

...


[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.393, twas_p=0.171); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/logtg/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/logtg/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/logtg/n05a_ki_twas_full_results.xlsx

Calculating Risk Score (Version 4: logp_expo) ...
  [weights] METABOLIC WEIGHTS DISABLED → uniform weight = 1.0
  [note] TWAS coverage: 960

100%|██████████| 70/70 [00:02<00:00, 31.53it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.089, twas_p=0.327); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/nonhdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/nonhdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/nonhdl/n05a_ki_twas_full_results.xlsx

Calculating Risk Score (Version 4: logp_expo) ...
  [weights] METABOLIC WEIGHTS DISABLED → uniform weight = 1.0
  [note] TWAS coverage: 

100%|██████████| 70/70 [00:02<00:00, 32.25it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.367, twas_p=0.21); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/tc/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/tc/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp_noweights/tc/n05a_ki_twas_full_results.xlsx

Calculating Risk Score (Version 4: logp_expo) ...
  [weights] METABOLIC WEIGHTS DISABLED → uniform weight = 1.0
  [note] TWAS coverage: 960/960 (100%

# Downstream

In [ ]:
# ======================================================================
# DOWNSTREAM ANALYSIS  --  N05A Ki→DDD→TWAS pipeline outputs only
# ----------------------------------------------------------------------
# Consumes ONLY the files already produced by the attached pipeline:
#     <root>/<trait>/metabolic_risk_score_per_drug*.csv   (per-drug risk)
#     <root>/<trait>/n05a_ki_twas_full_results.csv        (long format)
#
# Compares WITH-weights vs WITHOUT-weights (uniform 1.0) for:
#     V1_original   : no TWAS scaling
#     V4_logp_expo  : exp(beta * -log10(p)) scaling
# across lipid traits: ldl, hdl, logtg, nonhdl, tc
#
# Produces:
#   - rank concordance (Spearman / Kendall) with vs without weights
#   - drug rank-movement & score-change tables
#   - consensus vs divergent high-risk drugs
#   - cross-trait risk heatmaps (with & without weights)
#   - base_score vs final risk correlation (TWAS influence)
#   - strongest TWAS-significant strong-binder hits
#   - cross-trait consensus ranking (mean rank across traits)
#   - ONE very detailed downstream_summary.txt
# Run as a single Colab cell.  Requires: pandas numpy matplotlib (scipy optional)
# ======================================================================

import os
import glob
import datetime
import numpy as np
import pandas as pd

# ---- optional deps (all wrapped) ----
try:
    from scipy.stats import spearmanr, kendalltau
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _HAVE_MPL = True
except Exception:
    _HAVE_MPL = False


# ======================================================================
# CONFIG
# ======================================================================
PIPE_ROOT = "/content/pipeline_output"

VERSIONS = {
    "V1_original": {
        "withW": os.path.join(PIPE_ROOT, "n05a_ki_twas_log"),
        "noW":   os.path.join(PIPE_ROOT, "n05a_ki_twas_log_noweights"),
        "desc":  "Version 1 (no TWAS scaling; boost = 1 + f*max|z|)",
    },
    "V4_logp_expo": {
        "withW": os.path.join(PIPE_ROOT, "n05a_ki_twas_log_v3_logp"),
        "noW":   os.path.join(PIPE_ROOT, "n05a_ki_twas_log_v3_logp_noweights"),
        "desc":  "Version 4 (per-receptor TWAS scaling exp(beta*-log10 p))",
    },
}

TRAITS = ["ldl", "hdl", "logtg", "nonhdl", "tc"]

TOP_N          = 10     # top drugs to report per trait
CONSENSUS_N    = 8      # set size for consensus/divergence comparison
MOVERS_N       = 12     # top rank movers to list
TWAS_HITS_N    = 20     # top TWAS-significant strong-binder hits

OUT_DIR   = os.path.join(PIPE_ROOT, "downstream_analysis")
FIG_DIR   = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

REF_DRUG = "chlorpromazine"   # normalisation reference used by the pipeline


# ======================================================================
# Robust loaders
# ======================================================================
def _read_csv(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
        except Exception:
            break
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", low_memory=False)
    except Exception:
        return None


def find_risk_csv(folder):
    """Locate the per-drug risk CSV regardless of the weight-mode suffix."""
    if not os.path.isdir(folder):
        return None
    preferred = [
        "metabolic_risk_score_per_drug.csv",
        "metabolic_risk_score_per_drug_uniform_weights.csv",
        "metabolic_risk_score_per_drug_with_metabolic_weights.csv",
    ]
    for name in preferred:
        p = os.path.join(folder, name)
        if os.path.isfile(p):
            return p
    hits = sorted(glob.glob(os.path.join(folder, "metabolic_risk_score_per_drug*.csv")))
    return hits[0] if hits else None


def find_full_csv(folder):
    p = os.path.join(folder, "n05a_ki_twas_full_results.csv")
    return p if os.path.isfile(p) else None


# ======================================================================
# Build tidy master tables from outputs
# ======================================================================
load_log = []   # (version, weight_mode, trait, kind, status, path)

risk_rows = []
full_rows = []

for ver, cfg in VERSIONS.items():
    for wmode in ("withW", "noW"):
        root = cfg[wmode]
        for trait in TRAITS:
            folder = os.path.join(root, trait)

            rp = find_risk_csv(folder)
            if rp:
                df = _read_csv(rp)
                if df is not None and not df.empty:
                    df = df.copy()
                    df["version"] = ver
                    df["weight_mode"] = "with_weights" if wmode == "withW" else "uniform_weights"
                    df["trait"] = trait
                    risk_rows.append(df)
                    load_log.append((ver, wmode, trait, "risk", "OK", rp))
                else:
                    load_log.append((ver, wmode, trait, "risk", "EMPTY/UNREADABLE", rp))
            else:
                load_log.append((ver, wmode, trait, "risk", "MISSING", folder))

            fp = find_full_csv(folder)
            if fp:
                fdf = _read_csv(fp)
                if fdf is not None and not fdf.empty:
                    fdf = fdf.copy()
                    fdf["version"] = ver
                    fdf["weight_mode"] = "with_weights" if wmode == "withW" else "uniform_weights"
                    fdf["trait"] = trait
                    full_rows.append(fdf)
                    load_log.append((ver, wmode, trait, "full", "OK", fp))
                else:
                    load_log.append((ver, wmode, trait, "full", "EMPTY/UNREADABLE", fp))
            else:
                load_log.append((ver, wmode, trait, "full", "MISSING", folder))

risk_all = pd.concat(risk_rows, ignore_index=True) if risk_rows else pd.DataFrame()
full_all = pd.concat(full_rows, ignore_index=True) if full_rows else pd.DataFrame()

# normalise a couple of column dtypes we rely on
for col in ("risk_rank", "metabolic_risk_score", "base_score",
            "max_abs_twas_z", "n_twas_sig_metabolic", "risk_vs_reference"):
    if col in risk_all.columns:
        risk_all[col] = pd.to_numeric(risk_all[col], errors="coerce")

if not risk_all.empty:
    risk_all["drug_key"] = risk_all["drug"].astype(str).str.strip().str.lower()

if not full_all.empty:
    for col in ("Ki_nM", "affinity_dose_score", "twas_z", "twas_p"):
        if col in full_all.columns:
            full_all[col] = pd.to_numeric(full_all[col], errors="coerce")
    for col in ("strong_binder", "twas_significant"):
        if col in full_all.columns:
            full_all[col] = full_all[col].astype(str).str.lower().isin(
                ["true", "1", "1.0", "yes"]
            )


# ======================================================================
# Helpers for stats
# ======================================================================
def spearman_kendall(x, y):
    """Return (spearman_rho, spearman_p, kendall_tau, kendall_p) robustly."""
    x = pd.Series(x).astype(float)
    y = pd.Series(y).astype(float)
    ok = x.notna() & y.notna()
    x, y = x[ok], y[ok]
    if len(x) < 3 or x.nunique() < 2 or y.nunique() < 2:
        return (np.nan, np.nan, np.nan, np.nan)
    if _HAVE_SCIPY:
        rho, rp = spearmanr(x, y)
        tau, tp = kendalltau(x, y)
        return (float(rho), float(rp), float(tau), float(tp))
    rho = float(x.corr(y, method="spearman"))
    tau = float(x.corr(y, method="kendall"))
    return (rho, np.nan, tau, np.nan)


def get_risk(ver, wmode_label, trait):
    if risk_all.empty:
        return pd.DataFrame()
    return risk_all[
        (risk_all["version"] == ver)
        & (risk_all["weight_mode"] == wmode_label)
        & (risk_all["trait"] == trait)
    ].copy()


# ======================================================================
# ANALYSIS 1-3 : concordance, movement, consensus  (per version per trait)
# ======================================================================
concordance_rows = []
movement_rows = []
consensus_records = []   # dicts for summary

for ver in VERSIONS:
    for trait in TRAITS:
        w = get_risk(ver, "with_weights", trait)
        u = get_risk(ver, "uniform_weights", trait)
        if w.empty or u.empty:
            continue

        merged = w[["drug_key", "drug", "risk_rank", "metabolic_risk_score", "base_score",
                    "max_abs_twas_z"]].merge(
            u[["drug_key", "risk_rank", "metabolic_risk_score", "base_score",
               "max_abs_twas_z"]],
            on="drug_key", suffixes=("_with", "_without")
        )
        if merged.empty:
            continue

        rho, rp, tau, tp = spearman_kendall(
            merged["risk_rank_with"], merged["risk_rank_without"]
        )
        concordance_rows.append({
            "version": ver, "trait": trait, "n_drugs": len(merged),
            "spearman_rho": rho, "spearman_p": rp,
            "kendall_tau": tau, "kendall_p": tp,
        })

        m = merged.copy()
        # lower rank number = higher risk; positive rank_change => moved UP (riskier) without weights
        m["rank_change"] = m["risk_rank_with"] - m["risk_rank_without"]
        m["abs_rank_change"] = m["rank_change"].abs()
        m["score_change"] = (m["metabolic_risk_score_without"]
                             - m["metabolic_risk_score_with"])
        m["version"] = ver
        m["trait"] = trait
        movement_rows.append(m)

        # consensus / divergence (top CONSENSUS_N by smallest rank)
        top_w = set(w.nsmallest(CONSENSUS_N, "risk_rank")["drug_key"])
        top_u = set(u.nsmallest(CONSENSUS_N, "risk_rank")["drug_key"])
        name_map = dict(zip(w["drug_key"], w["drug"]))
        name_map.update(dict(zip(u["drug_key"], u["drug"])))

        def _names(keys):
            return sorted(name_map.get(k, k) for k in keys)

        consensus_records.append({
            "version": ver, "trait": trait,
            "consensus": _names(top_w & top_u),
            "only_with": _names(top_w - top_u),
            "only_without": _names(top_u - top_w),
        })

concordance_df = pd.DataFrame(concordance_rows)
movement_df = pd.concat(movement_rows, ignore_index=True) if movement_rows else pd.DataFrame()

if not concordance_df.empty:
    concordance_df.to_csv(os.path.join(OUT_DIR, "rank_concordance.csv"), index=False)
if not movement_df.empty:
    movement_df.to_csv(os.path.join(OUT_DIR, "rank_movement.csv"), index=False)


# ======================================================================
# ANALYSIS 4 : base_score vs final risk (how much TWAS moves the needle)
# ======================================================================
twas_influence_rows = []
for ver in VERSIONS:
    for wmode in ("with_weights", "uniform_weights"):
        for trait in TRAITS:
            d = get_risk(ver, wmode, trait)
            if d.empty or "base_score" not in d.columns:
                continue
            corr = np.nan
            if d["base_score"].notna().sum() >= 3:
                corr = float(d["base_score"].corr(d["metabolic_risk_score"],
                                                  method="pearson"))
            twas_influence_rows.append({
                "version": ver, "weight_mode": wmode, "trait": trait,
                "n_drugs": int(d["drug"].nunique()),
                "corr_base_vs_final": corr,
                "mean_max_abs_twas_z": float(d.get("max_abs_twas_z",
                                             pd.Series(dtype=float)).mean()),
            })
twas_influence_df = pd.DataFrame(twas_influence_rows)
if not twas_influence_df.empty:
    twas_influence_df.to_csv(os.path.join(OUT_DIR, "twas_influence.csv"), index=False)


# ======================================================================
# ANALYSIS 5 : cross-trait consensus ranking (mean rank across traits)
# ======================================================================
def cross_trait_consensus(ver, wmode):
    d = risk_all[(risk_all["version"] == ver)
                 & (risk_all["weight_mode"] == wmode)]
    if d.empty:
        return pd.DataFrame()
    g = (d.groupby("drug")
           .agg(mean_rank=("risk_rank", "mean"),
                mean_score=("metabolic_risk_score", "mean"),
                n_traits=("trait", "nunique"),
                min_rank=("risk_rank", "min"),
                max_rank=("risk_rank", "max"))
           .reset_index()
           .sort_values("mean_rank"))
    return g

consensus_cross = {}
for ver in VERSIONS:
    for wmode in ("with_weights", "uniform_weights"):
        g = cross_trait_consensus(ver, wmode)
        consensus_cross[(ver, wmode)] = g
        if not g.empty:
            g.to_csv(os.path.join(
                OUT_DIR, f"cross_trait_consensus_{ver}_{wmode}.csv"), index=False)


# ======================================================================
# ANALYSIS 6 : TWAS-significant strong-binder hits (from long format)
# ======================================================================
twas_hits = pd.DataFrame()
if not full_all.empty and {"twas_significant", "affinity_dose_score"}.issubset(full_all.columns):
    twas_hits = full_all[
        full_all["twas_significant"].fillna(False)
        & full_all["affinity_dose_score"].notna()
    ].copy()
    keep_cols = ["version", "weight_mode", "trait", "drug", "gene", "receptor",
                 "Ki_nM", "affinity_dose_score", "twas_z", "twas_p",
                 "strong_binder"]
    keep_cols = [c for c in keep_cols if c in twas_hits.columns]
    twas_hits = twas_hits[keep_cols].sort_values(
        ["twas_p", "affinity_dose_score"], ascending=[True, False]
    )
    if not twas_hits.empty:
        twas_hits.to_csv(os.path.join(OUT_DIR, "twas_significant_hits.csv"), index=False)


# ======================================================================
# ANALYSIS 7 : heatmaps (optional, saved to disk)
# ======================================================================
def build_heatmap_df(ver, wmode):
    d = risk_all[(risk_all["version"] == ver)
                 & (risk_all["weight_mode"] == wmode)]
    if d.empty:
        return pd.DataFrame()
    return d.pivot_table(index="drug", columns="trait",
                         values="metabolic_risk_score", aggfunc="first")

heatmap_paths = []
if _HAVE_MPL:
    for ver in VERSIONS:
        for wmode in ("with_weights", "uniform_weights"):
            hm = build_heatmap_df(ver, wmode)
            if hm.empty:
                continue
            hm = hm.reindex(columns=[t for t in TRAITS if t in hm.columns])
            hm = hm.loc[hm.mean(axis=1).sort_values(ascending=False).index]
            try:
                fig_h = max(4, 0.32 * len(hm) + 2)
                fig, ax = plt.subplots(figsize=(8, fig_h))
                data = hm.values.astype(float)
                im = ax.imshow(data, aspect="auto", cmap="RdYlGn_r")
                ax.set_xticks(range(hm.shape[1]))
                ax.set_xticklabels(hm.columns, rotation=45, ha="right")
                ax.set_yticks(range(hm.shape[0]))
                ax.set_yticklabels(hm.index, fontsize=7)
                ax.set_title(f"Risk score  {ver}  [{wmode}]")
                fig.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
                fig.tight_layout()
                fp = os.path.join(FIG_DIR, f"heatmap_{ver}_{wmode}.png")
                fig.savefig(fp, dpi=130)
                plt.close(fig)
                heatmap_paths.append(fp)
            except Exception as e:
                print(f"  [warn] heatmap failed {ver}/{wmode}: {e}")

# rank-movement dot plots (V4 tends to be the main figure)
mover_fig_paths = []
if _HAVE_MPL and not movement_df.empty:
    for ver in VERSIONS:
        sub = movement_df[movement_df["version"] == ver]
        if sub.empty:
            continue
        agg = (sub.groupby("drug")["rank_change"].mean()
                  .sort_values())
        if agg.empty:
            continue
        try:
            fig_h = max(4, 0.3 * len(agg) + 2)
            fig, ax = plt.subplots(figsize=(8, fig_h))
            colors = ["#2c7fb8" if v >= 0 else "#d95f0e" for v in agg.values]
            ax.hlines(y=range(len(agg)), xmin=0, xmax=agg.values, color=colors)
            ax.plot(agg.values, range(len(agg)), "o", color="black", ms=3)
            ax.set_yticks(range(len(agg)))
            ax.set_yticklabels(agg.index, fontsize=7)
            ax.axvline(0, color="grey", lw=0.8)
            ax.set_xlabel("mean rank change (with − without)  |  +ve = riskier without weights")
            ax.set_title(f"Rank movement (weights removed)  {ver}")
            fig.tight_layout()
            fp = os.path.join(FIG_DIR, f"rank_movement_{ver}.png")
            fig.savefig(fp, dpi=130)
            plt.close(fig)
            mover_fig_paths.append(fp)
        except Exception as e:
            print(f"  [warn] mover plot failed {ver}: {e}")


# ======================================================================
# WRITE THE DETAILED SUMMARY TXT
# ======================================================================
def _fmt(x, spec=".3f"):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "NA"
        return format(x, spec)
    except Exception:
        return str(x)

L = []
bar = "=" * 82
sub = "-" * 82
L.append(bar)
L.append("N05A Ki→DDD→TWAS  —  DOWNSTREAM ANALYSIS  (WITH vs WITHOUT metabolic weights)")
L.append(bar)
L.append(f"Generated             : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
L.append(f"Pipeline output root  : {PIPE_ROOT}")
L.append(f"Analysis output dir   : {OUT_DIR}")
L.append(f"Traits analysed       : {', '.join(TRAITS)}")
L.append(f"Versions compared     : {', '.join(VERSIONS.keys())}")
L.append(f"scipy available       : {_HAVE_SCIPY}   |   matplotlib available : {_HAVE_MPL}")
L.append(f"Reference drug (norm) : {REF_DRUG}")
L.append("")
L.append("Interpretation notes:")
L.append("  * risk_rank: 1 = highest predicted risk. Scores normalised so reference=100.")
L.append("  * rank_change = rank_with − rank_without; POSITIVE => drug becomes RISKIER")
L.append("    (moves toward rank 1) when literature weights are removed.")
L.append("  * 'with_weights' = literature METABOLIC_WEIGHTS; 'uniform_weights' = 1.0 each.")
L.append("")

# ---- file discovery log ----
L.append(bar); L.append("0. INPUT FILE DISCOVERY"); L.append(bar)
n_ok = sum(1 for r in load_log if r[4] == "OK")
n_bad = len(load_log) - n_ok
L.append(f"Files located OK: {n_ok}   |   missing/unreadable: {n_bad}")
missing = [r for r in load_log if r[4] != "OK"]
if missing:
    L.append("Problems:")
    for ver, wmode, trait, kind, status, path in missing:
        L.append(f"  [{status:<16}] {ver:<13} {wmode:<5} {trait:<7} {kind:<5} -> {path}")
L.append("")

if risk_all.empty:
    L.append("!! No per-drug risk CSVs were loaded. Nothing further to analyse.")
else:
    # =========================================================
    # 1. RANK CONCORDANCE
    # =========================================================
    L.append(bar); L.append("1. RANK CONCORDANCE  (with-weights vs uniform-weights)"); L.append(bar)
    if concordance_df.empty:
        L.append("  No paired traits available for concordance.")
    else:
        L.append(f"  {'version':<14}{'trait':<8}{'n':>4}{'spearman_rho':>14}"
                 f"{'p':>10}{'kendall_tau':>13}{'p':>10}")
        L.append("  " + sub)
        for ver in VERSIONS:
            sub_c = concordance_df[concordance_df["version"] == ver]
            for _, r in sub_c.iterrows():
                L.append(f"  {ver:<14}{r['trait']:<8}{int(r['n_drugs']):>4}"
                         f"{_fmt(r['spearman_rho']):>14}{_fmt(r['spearman_p']):>10}"
                         f"{_fmt(r['kendall_tau']):>13}{_fmt(r['kendall_p']):>10}")
            if not sub_c.empty:
                L.append(f"  {ver:<14}{'MEAN':<8}{'':>4}"
                         f"{_fmt(sub_c['spearman_rho'].mean()):>14}{'':>10}"
                         f"{_fmt(sub_c['kendall_tau'].mean()):>13}{'':>10}")
                L.append("")
        L.append("  Guide: rho>=0.9 near-identical ranking; 0.7-0.9 moderate reshuffle;")
        L.append("         <0.7 substantial re-ranking driven by removing weights.")
        # flag low-concordance trait/version combos
        low = concordance_df[concordance_df["spearman_rho"] < 0.7]
        if not low.empty:
            L.append("")
            L.append("  ** Low-concordance (rho<0.7) combinations (weights matter most):")
            for _, r in low.iterrows():
                L.append(f"       {r['version']} / {r['trait']}: rho={_fmt(r['spearman_rho'])}")
    L.append("")

    # =========================================================
    # 2. TOP DRUGS PER TRAIT (side by side)
    # =========================================================
    L.append(bar); L.append(f"2. TOP {TOP_N} DRUGS PER TRAIT  (with vs without weights)"); L.append(bar)
    for ver in VERSIONS:
        L.append(f"### {ver}  —  {VERSIONS[ver]['desc']}")
        for trait in TRAITS:
            w = get_risk(ver, "with_weights", trait)
            u = get_risk(ver, "uniform_weights", trait)
            if w.empty and u.empty:
                continue
            L.append(f"  Trait: {trait.upper()}")
            wl = (w.nsmallest(TOP_N, "risk_rank")[["risk_rank", "drug",
                  "metabolic_risk_score"]].values.tolist() if not w.empty else [])
            ul = (u.nsmallest(TOP_N, "risk_rank")[["risk_rank", "drug",
                  "metabolic_risk_score"]].values.tolist() if not u.empty else [])
            L.append(f"    {'#':>2}  {'WITH weights':<30}{'score':>8}   | "
                     f" {'WITHOUT weights':<30}{'score':>8}")
            for i in range(TOP_N):
                lw = wl[i] if i < len(wl) else None
                lu = ul[i] if i < len(ul) else None
                lw_s = (f"{str(lw[1])[:28]:<30}{_fmt(lw[2],'.1f'):>8}"
                        if lw else " " * 38)
                lu_s = (f"{str(lu[1])[:28]:<30}{_fmt(lu[2],'.1f'):>8}"
                        if lu else "")
                L.append(f"    {i+1:>2}  {lw_s}   |  {lu_s}")
            L.append("")
        L.append("")

    # =========================================================
    # 3. CONSENSUS vs DIVERGENT HIGH-RISK DRUGS
    # =========================================================
    L.append(bar)
    L.append(f"3. CONSENSUS vs DIVERGENT HIGH-RISK (top {CONSENSUS_N} per mode)")
    L.append(bar)
    for rec in consensus_records:
        L.append(f"  {rec['version']} / {rec['trait'].upper()}")
        L.append(f"    consensus high-risk : {', '.join(rec['consensus']) or '(none)'}")
        L.append(f"    high ONLY w/ weights: {', '.join(rec['only_with']) or '(none)'}")
        L.append(f"    high ONLY w/o weight: {', '.join(rec['only_without']) or '(none)'}")
        L.append("")
    L.append("")

    # =========================================================
    # 4. BIGGEST RANK MOVERS
    # =========================================================
    L.append(bar); L.append(f"4. BIGGEST RANK MOVERS (weights removed) — top {MOVERS_N}"); L.append(bar)
    if movement_df.empty:
        L.append("  No movement data available.")
    else:
        for ver in VERSIONS:
            s = movement_df[movement_df["version"] == ver]
            if s.empty:
                continue
            L.append(f"### {ver}")
            # aggregate mean over traits per drug
            agg = (s.groupby("drug")
                     .agg(mean_rank_change=("rank_change", "mean"),
                          mean_score_change=("score_change", "mean"),
                          n_traits=("trait", "nunique"),
                          mean_max_twas_z=("max_abs_twas_z_without", "mean"))
                     .reset_index())
            agg["abs_change"] = agg["mean_rank_change"].abs()
            agg = agg.sort_values("abs_change", ascending=False).head(MOVERS_N)
            L.append(f"    {'drug':<28}{'Δrank(mean)':>12}{'Δscore(mean)':>14}"
                     f"{'traits':>8}{'max|z|':>9}")
            for _, r in agg.iterrows():
                direction = "↑riskier" if r["mean_rank_change"] > 0 else (
                            "↓safer" if r["mean_rank_change"] < 0 else "=")
                L.append(f"    {str(r['drug'])[:27]:<28}"
                         f"{_fmt(r['mean_rank_change'],'+.1f'):>12}"
                         f"{_fmt(r['mean_score_change'],'+.1f'):>14}"
                         f"{int(r['n_traits']):>8}"
                         f"{_fmt(r['mean_max_twas_z'],'.2f'):>9}  {direction}")
            L.append("")
    L.append("")

    # =========================================================
    # 5. CROSS-TRAIT CONSENSUS RANKING (mean rank over all traits)
    # =========================================================
    L.append(bar); L.append("5. CROSS-TRAIT CONSENSUS RANKING (mean risk_rank across traits)"); L.append(bar)
    for ver in VERSIONS:
        for wmode in ("with_weights", "uniform_weights"):
            g = consensus_cross.get((ver, wmode), pd.DataFrame())
            if g.empty:
                continue
            L.append(f"### {ver}  [{wmode}]  (lower mean_rank = more consistently high-risk)")
            L.append(f"    {'#':>2}  {'drug':<28}{'mean_rank':>10}{'mean_score':>11}"
                     f"{'n_traits':>9}{'rank_range':>12}")
            for i, (_, r) in enumerate(g.head(TOP_N).iterrows(), start=1):
                rng = f"{int(r['min_rank'])}-{int(r['max_rank'])}"
                L.append(f"    {i:>2}  {str(r['drug'])[:27]:<28}"
                         f"{_fmt(r['mean_rank'],'.2f'):>10}"
                         f"{_fmt(r['mean_score'],'.1f'):>11}"
                         f"{int(r['n_traits']):>9}{rng:>12}")
            L.append("")
    L.append("")

    # =========================================================
    # 6. TWAS INFLUENCE (base_score vs final risk)
    # =========================================================
    L.append(bar); L.append("6. TWAS INFLUENCE  (Pearson corr base_score vs final risk)"); L.append(bar)
    if twas_influence_df.empty:
        L.append("  No base_score column available.")
    else:
        L.append(f"  {'version':<14}{'weight_mode':<18}{'trait':<8}"
                 f"{'corr':>8}{'mean_max|z|':>13}")
        L.append("  " + sub)
        for _, r in twas_influence_df.iterrows():
            L.append(f"  {r['version']:<14}{r['weight_mode']:<18}{r['trait']:<8}"
                     f"{_fmt(r['corr_base_vs_final'],'.3f'):>8}"
                     f"{_fmt(r['mean_max_abs_twas_z'],'.3f'):>13}")
        L.append("")
        L.append("  corr≈1.0 => TWAS boost barely reshuffles; lower corr => TWAS")
        L.append("  meaningfully changes the ordering relative to raw affinity×dose.")
    L.append("")

    # =========================================================
    # 7. TOP TWAS-SIGNIFICANT STRONG-BINDER HITS
    # =========================================================
    L.append(bar)
    L.append(f"7. TOP TWAS-SIGNIFICANT HITS (p<0.05) — up to {TWAS_HITS_N} per version")
    L.append(bar)
    if twas_hits.empty:
        L.append("  No TWAS-significant rows found in the long-format outputs.")
        L.append("  (Check that TWAS files loaded and p-values were present.)")
    else:
        for ver in VERSIONS:
            sub_h = twas_hits[twas_hits["version"] == ver]
            # de-duplicate identical drug×gene×trait across weight modes
            if not sub_h.empty and {"drug", "gene", "trait"}.issubset(sub_h.columns):
                sub_h = sub_h.drop_duplicates(subset=["trait", "drug", "gene"])
            sub_h = sub_h.sort_values(
                ["twas_p", "affinity_dose_score"], ascending=[True, False]
            ).head(TWAS_HITS_N)
            if sub_h.empty:
                continue
            L.append(f"### {ver}")
            L.append(f"    {'trait':<7}{'drug':<20}{'gene':<8}{'Ki_nM':>10}"
                     f"{'score':>12}{'twas_z':>9}{'twas_p':>11}")
            for _, r in sub_h.iterrows():
                L.append(f"    {str(r.get('trait',''))[:6]:<7}"
                         f"{str(r.get('drug',''))[:19]:<20}"
                         f"{str(r.get('gene',''))[:7]:<8}"
                         f"{_fmt(r.get('Ki_nM'),'.3g'):>10}"
                         f"{_fmt(r.get('affinity_dose_score'),'.4g'):>12}"
                         f"{_fmt(r.get('twas_z'),'.3f'):>9}"
                         f"{_fmt(r.get('twas_p'),'.3g'):>11}")
            L.append("")

        # which metabolic genes recur among significant hits
        if "gene" in twas_hits.columns:
            gene_counts = (twas_hits.dropna(subset=["gene"])
                           .groupby("gene")["drug"].nunique()
                           .sort_values(ascending=False).head(15))
            if not gene_counts.empty:
                L.append("  Most recurrent TWAS-significant receptor genes (by #drugs):")
                for g, n in gene_counts.items():
                    L.append(f"       {g:<10} {int(n)} drug(s)")
                L.append("")
    L.append("")

    # =========================================================
    # 8. HEADLINE TAKEAWAYS (auto-generated flags)
    # =========================================================
    L.append(bar); L.append("8. HEADLINE TAKEAWAYS (auto-generated)"); L.append(bar)
    if not concordance_df.empty:
        for ver in VERSIONS:
            sc = concordance_df[concordance_df["version"] == ver]
            if sc.empty:
                continue
            mrho = sc["spearman_rho"].mean()
            verdict = ("robust to weighting" if mrho >= 0.9 else
                       "moderately sensitive to weighting" if mrho >= 0.7 else
                       "HIGHLY sensitive to weighting")
            L.append(f"  * {ver}: mean Spearman ρ = {_fmt(mrho)} → ranking is {verdict}.")
    # consistently-riskiest drug across everything (uniform, cross-trait)
    for ver in VERSIONS:
        g = consensus_cross.get((ver, "uniform_weights"), pd.DataFrame())
        if not g.empty:
            top = g.iloc[0]
            L.append(f"  * {ver} [uniform]: most consistently high-risk drug = "
                     f"{top['drug']} (mean_rank={_fmt(top['mean_rank'],'.2f')}).")
        gw = consensus_cross.get((ver, "with_weights"), pd.DataFrame())
        if not gw.empty:
            topw = gw.iloc[0]
            L.append(f"  * {ver} [weights]: most consistently high-risk drug = "
                     f"{topw['drug']} (mean_rank={_fmt(topw['mean_rank'],'.2f')}).")
    # biggest single mover overall (V4 if present else V1)
    if not movement_df.empty:
        for ver in VERSIONS:
            s = movement_df[movement_df["version"] == ver]
            if s.empty:
                continue
            agg = s.groupby("drug")["rank_change"].mean()
            if agg.empty:
                continue
            up = agg.idxmax(); dn = agg.idxmin()
            L.append(f"  * {ver}: removing weights most INCREASES risk-rank of "
                     f"'{up}' (+{_fmt(agg.max(),'.1f')}) and most DECREASES '{dn}' "
                     f"({_fmt(agg.min(),'.1f')}).")
    L.append("")

# ---- artefact index ----
L.append(bar); L.append("9. GENERATED ARTEFACTS"); L.append(bar)
for f in ["rank_concordance.csv", "rank_movement.csv", "twas_influence.csv",
          "twas_significant_hits.csv"]:
    p = os.path.join(OUT_DIR, f)
    L.append(f"  {'[OK]' if os.path.isfile(p) else '[--]'} {p}")
for (ver, wmode), g in consensus_cross.items():
    p = os.path.join(OUT_DIR, f"cross_trait_consensus_{ver}_{wmode}.csv")
    if os.path.isfile(p):
        L.append(f"  [OK] {p}")
for p in heatmap_paths + mover_fig_paths:
    L.append(f"  [FIG] {p}")
L.append("")
L.append(bar); L.append("END OF DOWNSTREAM SUMMARY"); L.append(bar)

summary_text = "\n".join(L)
summary_path = os.path.join(OUT_DIR, "downstream_summary.txt")
with open(summary_path, "w", encoding="utf-8") as fh:
    fh.write(summary_text)

# ======================================================================
# Console echo
# ======================================================================
print(summary_text)
print("\n" + "#" * 82)
print(f"# Detailed summary written to: {summary_path}")
print(f"# All analysis CSVs + figures in: {OUT_DIR}")
print("#" * 82)

# (optional) copy analysis back to Drive – uncomment if mounted
# !cp -r "/content/pipeline_output/downstream_analysis" "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS/"

N05A Ki→DDD→TWAS  —  DOWNSTREAM ANALYSIS  (WITH vs WITHOUT metabolic weights)
Generated             : 2026-07-15 14:27:12
Pipeline output root  : /content/pipeline_output
Analysis output dir   : /content/pipeline_output/downstream_analysis
Traits analysed       : ldl, hdl, logtg, nonhdl, tc
Versions compared     : V1_original, V4_logp_expo
scipy available       : True   |   matplotlib available : True
Reference drug (norm) : chlorpromazine

Interpretation notes:
  * risk_rank: 1 = highest predicted risk. Scores normalised so reference=100.
  * rank_change = rank_with − rank_without; POSITIVE => drug becomes RISKIER
    (moves toward rank 1) when literature weights are removed.
  * 'with_weights' = literature METABOLIC_WEIGHTS; 'uniform_weights' = 1.0 each.

0. INPUT FILE DISCOVERY
Files located OK: 40   |   missing/unreadable: 0

1. RANK CONCORDANCE  (with-weights vs uniform-weights)
  version       trait      n  spearman_rho         p  kendall_tau         p
  -------------------------

# Stage 2 Downstream

In [ ]:
# ======================================================================
# DOWNSTREAM ANALYSIS — STAGE 2 (MECHANISTIC)
# ----------------------------------------------------------------------
# Consumes ONLY files already produced by the attached pipeline:
#     <root>/<trait>/n05a_ki_twas_full_results.csv        (long format)
#     <root>/<trait>/metabolic_risk_score_per_drug*.csv   (per-drug risk)
#
# Focus = MECHANISTIC TRANSPARENCY & CLINICAL INTERPRETATION for
# V4 (logp_expo) WITH-weights vs WITHOUT-weights (uniform), across
# lipid traits: ldl, hdl, logtg, nonhdl, tc
#
# Analyses:
#   1. Per-receptor GLOBAL contribution (with vs uniform), per trait + pooled
#   2. TWAS signal vs literature weight  → novel-mechanism candidates
#   3. Discordance vs clinical metabolic-liability literature
#   4. Leave-one-gene-out sensitivity (ADRB1 + top TWAS genes) → rank shifts
#   5. Drugs most sensitive to TWAS scaling (V1 no-scaling vs V4 logp_expo)
#   6. Receptor co-occurrence in top high-risk drugs
#   + reconstruction validation vs saved risk CSVs
#   + ONE very detailed downstream_stage2_summary.txt
#
# Run as a single Colab cell.  Requires: pandas numpy (scipy/matplotlib optional)
# ======================================================================

import os
import glob
import datetime
import itertools
import numpy as np
import pandas as pd

try:
    from scipy.stats import spearmanr
    _HAVE_SCIPY = True
except Exception:
    _HAVE_SCIPY = False

try:
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    _HAVE_MPL = True
except Exception:
    _HAVE_MPL = False


# ======================================================================
# CONFIG  --  must match the pipeline's V4 (logp_expo) settings
# ======================================================================
PIPE_ROOT = "/content/pipeline_output"

# roots for each mode / version
V4_WITHW = os.path.join(PIPE_ROOT, "n05a_ki_twas_log_v3_logp")
V4_NOW   = os.path.join(PIPE_ROOT, "n05a_ki_twas_log_v3_logp_noweights")
V1_WITHW = os.path.join(PIPE_ROOT, "n05a_ki_twas_log")            # for TWAS-scaling sensitivity
V1_NOW   = os.path.join(PIPE_ROOT, "n05a_ki_twas_log_noweights")

TRAITS = ["ldl", "hdl", "logtg", "nonhdl", "tc"]

# V4 scoring parameters (mirror the pipeline exactly)
LOGP_BETA          = 0.045
MAX_NEGLOG10_P     = 25.0
SECONDARY_BOOST    = 0.02
TWAS_SIG_BONUS     = 0.10
TWAS_P_SIGNIFICANT = 0.05
REF_DRUG           = "chlorpromazine"

TOP_RECEPTORS   = 15
TOP_DRUGS_NET   = 10
NOVEL_Z_MIN     = 4.0    # |z| threshold for "strong" TWAS
NOVEL_W_MAX     = 0.30   # literature weight below this = "low weight"
LEAVE_OUT_GENES = ["ADRB1", "HRH1", "HTR2C", "CHRM3"]  # focus genes for sensitivity

OUT_DIR = os.path.join(PIPE_ROOT, "downstream_stage2")
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


# ======================================================================
# Literature metabolic weights (copied from the pipeline)
# ======================================================================
METABOLIC_WEIGHTS = {
    "HRH1": 1.00, "HTR2C": 0.92, "CHRM3": 0.85, "HTR2A": 0.55,
    "ADRA1A": 0.48, "ADRA1B": 0.48, "HTR6": 0.42, "ADRA2A": 0.32,
    "ADRA2B": 0.30, "ADRA2C": 0.30, "CHRM1": 0.28, "CHRM4": 0.22,
    "CHRM5": 0.20, "HTR7": 0.25, "DRD3": 0.18, "DRD2": 0.12,
    "DRD4": 0.10, "HTR1A": 0.08, "ADRB1": 0.08, "ADRB2": 0.07,
    "SIGMAR1": 0.05, "SLC6A4": 0.06, "SLC6A2": 0.05, "SLC6A3": 0.04,
}

# Established clinical metabolic-liability rank (1 = highest risk).
# Widely-cited ordering; used only for a discordance comparison.
CLINICAL_RANK = {
    "clozapine": 1, "olanzapine": 2, "quetiapine": 3, "paliperidone": 4,
    "risperidone": 5, "asenapine": 6, "aripiprazole": 7, "amisulpride": 8,
    "ziprasidone": 9, "lurasidone": 10, "haloperidol": 11, "cariprazine": 12,
}


# ======================================================================
# Robust loaders
# ======================================================================
def _read_csv(path):
    for enc in ("utf-8", "utf-8-sig", "latin-1", "cp1252"):
        try:
            return pd.read_csv(path, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
        except Exception:
            break
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", low_memory=False)
    except Exception:
        return None


def find_risk_csv(folder):
    if not os.path.isdir(folder):
        return None
    for name in ("metabolic_risk_score_per_drug.csv",
                 "metabolic_risk_score_per_drug_uniform_weights.csv",
                 "metabolic_risk_score_per_drug_with_metabolic_weights.csv"):
        p = os.path.join(folder, name)
        if os.path.isfile(p):
            return p
    hits = sorted(glob.glob(os.path.join(folder,
                  "metabolic_risk_score_per_drug*.csv")))
    return hits[0] if hits else None


def find_full_csv(folder):
    p = os.path.join(folder, "n05a_ki_twas_full_results.csv")
    return p if os.path.isfile(p) else None


def load_long(root, trait):
    fp = find_full_csv(os.path.join(root, trait))
    if not fp:
        return None, fp, "MISSING"
    df = _read_csv(fp)
    if df is None or df.empty:
        return None, fp, "EMPTY/UNREADABLE"
    for c in ("affinity_dose_score", "twas_z", "twas_p", "Ki_nM"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df, fp, "OK"


def load_risk(root, trait):
    fp = find_risk_csv(os.path.join(root, trait))
    if not fp:
        return None, fp, "MISSING"
    df = _read_csv(fp)
    if df is None or df.empty:
        return None, fp, "EMPTY/UNREADABLE"
    for c in ("risk_rank", "metabolic_risk_score", "base_score",
              "max_abs_twas_z"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["drug_key"] = df["drug"].astype(str).str.strip().str.lower()
    return df, fp, "OK"


# ======================================================================
# Faithful reconstruction of per-receptor V4 contribution
#   affinity   = log1p(max(affinity_dose_score, 0))
#   twas_scale = exp(beta * min(-log10(p), cap))     (p missing -> scale 1)
#   weight     = METABOLIC_WEIGHTS[gene]  (with-weights)  |  1.0 (uniform)
#   contribution = affinity * weight * twas_scale
# ======================================================================
def reconstruct_contributions(long_df, use_weights):
    df = long_df.copy()
    df = df[df["gene"].notna() & df["affinity_dose_score"].notna()].copy()
    if df.empty:
        return df

    aff = np.log1p(df["affinity_dose_score"].clip(lower=0))

    if use_weights:
        w = df["gene"].map(METABOLIC_WEIGHTS).fillna(0.0)
    else:
        w = pd.Series(1.0, index=df.index)

    p_safe = df["twas_p"].clip(lower=1e-300)
    neglogp = (-np.log10(p_safe)).clip(upper=MAX_NEGLOG10_P)
    twas_scale = np.exp(LOGP_BETA * neglogp.fillna(0.0))

    df["affinity_t"]   = aff
    df["rec_weight"]   = w
    df["twas_scale"]   = twas_scale
    df["contribution"] = aff * w * twas_scale
    return df


def drug_scores_from_contrib(contrib_df, normalize=True):
    """Rebuild per-drug score mirroring V4 (base + secondary boost + sig bonus)."""
    if contrib_df.empty:
        return pd.DataFrame()
    base = (contrib_df.groupby("drug")["contribution"].sum()
            .reset_index().rename(columns={"contribution": "base_score"}))

    maxz = (contrib_df.groupby("drug")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
            .reset_index().rename(columns={"twas_z": "max_abs_twas_z"}))
    base = base.merge(maxz, on="drug", how="left")
    base["max_abs_twas_z"] = base["max_abs_twas_z"].fillna(0.0)

    sig = contrib_df[(contrib_df["twas_p"] < TWAS_P_SIGNIFICANT)
                     & (contrib_df["rec_weight"] > 0)]
    nsig = (sig.groupby("drug").size().reset_index(name="n_sig"))
    base = base.merge(nsig, on="drug", how="left")
    base["n_sig"] = base["n_sig"].fillna(0).astype(int)

    base["score"] = (base["base_score"]
                     * (1 + SECONDARY_BOOST * base["max_abs_twas_z"])
                     * (1 + TWAS_SIG_BONUS * base["n_sig"]))

    if normalize:
        rm = base["drug"].astype(str).str.lower() == REF_DRUG.lower()
        ref = base.loc[rm, "score"]
        ref_val = ref.values[0] if (not ref.empty and ref.values[0] > 0) \
                  else base["score"].max()
        if ref_val and ref_val > 0:
            base["score"] = base["score"] / ref_val * 100

    base = base[base["score"] != 0].copy()
    base["rank"] = base["score"].rank(ascending=False, method="min").astype(int)
    return base.sort_values("rank").reset_index(drop=True)


# ======================================================================
# LOAD everything
# ======================================================================
load_log = []
long_store = {}   # (mode, trait) -> long df   ; mode in {"withW","noW"}
risk_store = {}   # (version, mode, trait) -> risk df

for mode, root in (("withW", V4_WITHW), ("noW", V4_NOW)):
    for trait in TRAITS:
        df, fp, status = load_long(root, trait)
        load_log.append(("V4", mode, trait, "full", status, fp))
        if df is not None:
            long_store[(mode, trait)] = df

for ver, wroot, nroot in (("V4", V4_WITHW, V4_NOW), ("V1", V1_WITHW, V1_NOW)):
    for mode, root in (("withW", wroot), ("noW", nroot)):
        for trait in TRAITS:
            df, fp, status = load_risk(root, trait)
            load_log.append((ver, mode, trait, "risk", status, fp))
            if df is not None:
                risk_store[(ver, mode, trait)] = df


# ======================================================================
# ANALYSIS 1 : per-receptor GLOBAL contribution
# ======================================================================
global_rows = []
for mode in ("withW", "noW"):
    use_w = (mode == "withW")
    for trait in TRAITS:
        ldf = long_store.get((mode, trait))
        if ldf is None:
            continue
        c = reconstruct_contributions(ldf, use_weights=use_w)
        if c.empty:
            continue
        g = (c.groupby(["receptor", "gene"])["contribution"].sum()
               .reset_index())
        g["trait"] = trait
        g["mode"]  = "with_weights" if use_w else "uniform"
        global_rows.append(g)

global_contrib = pd.concat(global_rows, ignore_index=True) if global_rows else pd.DataFrame()
if not global_contrib.empty:
    global_contrib.to_csv(os.path.join(OUT_DIR, "global_receptor_contribution.csv"),
                          index=False)
    # pooled across traits
    global_pooled = (global_contrib.groupby(["mode", "gene"])["contribution"]
                     .sum().reset_index())
    global_pooled.to_csv(os.path.join(OUT_DIR,
                         "global_receptor_contribution_pooled.csv"), index=False)
else:
    global_pooled = pd.DataFrame()


# ======================================================================
# ANALYSIS 2 : TWAS signal vs literature weight  (novel candidates)
# ======================================================================
twas_weight_rows = []
for trait in TRAITS:
    ldf = long_store.get(("withW", trait))
    if ldf is None:
        continue
    g = (ldf.dropna(subset=["gene"])
            .groupby("gene")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else np.nan)
            .reset_index().rename(columns={"twas_z": "max_abs_twas_z"}))
    g["trait"] = trait
    twas_weight_rows.append(g)

twas_weight_df = pd.DataFrame()
novel_candidates = pd.DataFrame()
if twas_weight_rows:
    tw = pd.concat(twas_weight_rows, ignore_index=True)
    # best (max) |z| per gene across traits + trait where it peaks
    idx = tw.groupby("gene")["max_abs_twas_z"].idxmax().dropna()
    twas_weight_df = tw.loc[idx].copy()
    twas_weight_df["literature_weight"] = (
        twas_weight_df["gene"].map(METABOLIC_WEIGHTS).fillna(0.0))
    twas_weight_df = twas_weight_df.sort_values(
        "max_abs_twas_z", ascending=False).reset_index(drop=True)
    twas_weight_df.to_csv(os.path.join(OUT_DIR, "twas_vs_literature_weight.csv"),
                          index=False)
    novel_candidates = twas_weight_df[
        (twas_weight_df["max_abs_twas_z"] >= NOVEL_Z_MIN)
        & (twas_weight_df["literature_weight"] <= NOVEL_W_MAX)
    ].copy()

# scatter figure
if _HAVE_MPL and not twas_weight_df.empty:
    try:
        fig, ax = plt.subplots(figsize=(9, 6))
        ax.scatter(twas_weight_df["literature_weight"],
                   twas_weight_df["max_abs_twas_z"], s=45, alpha=0.75)
        for _, r in twas_weight_df.iterrows():
            if (r["max_abs_twas_z"] >= NOVEL_Z_MIN) or (r["literature_weight"] >= 0.5):
                ax.annotate(str(r["gene"]),
                            (r["literature_weight"], r["max_abs_twas_z"]),
                            fontsize=7, xytext=(3, 3), textcoords="offset points")
        ax.axhline(NOVEL_Z_MIN, color="red", ls="--", lw=0.8,
                   label=f"|z|={NOVEL_Z_MIN}")
        ax.axvline(NOVEL_W_MAX, color="grey", ls=":", lw=0.8)
        ax.set_xlabel("Literature metabolic weight")
        ax.set_ylabel("Max |TWAS z| across traits")
        ax.set_title("TWAS signal vs literature weight (V4, with-weights)")
        ax.legend()
        fig.tight_layout()
        fp = os.path.join(FIG_DIR, "twas_vs_literature_weight.png")
        fig.savefig(fp, dpi=130); plt.close(fig)
    except Exception as e:
        print(f"[warn] scatter failed: {e}")


# ======================================================================
# ANALYSIS 3 : discordance vs clinical literature
# ======================================================================
def build_discordance(mode_label):
    rows = []
    for trait in TRAITS:
        rd = risk_store.get(("V4", "withW" if mode_label == "with_weights" else "noW", trait))
        if rd is None:
            continue
        d = rd[["drug_key", "drug", "risk_rank"]].copy()
        d["trait"] = trait
        rows.append(d)
    if not rows:
        return pd.DataFrame()
    allr = pd.concat(rows, ignore_index=True)
    agg = (allr.groupby(["drug_key", "drug"])["risk_rank"]
           .mean().reset_index().rename(columns={"risk_rank": "mean_model_rank"}))
    agg["model_rank"] = agg["mean_model_rank"].rank(method="min").astype(int)
    agg["clinical_rank"] = agg["drug_key"].map(CLINICAL_RANK)
    agg = agg.dropna(subset=["clinical_rank"]).copy()
    agg["clinical_rank"] = agg["clinical_rank"].astype(int)
    agg["discordance"] = agg["model_rank"] - agg["clinical_rank"]
    agg["mode"] = mode_label
    return agg.sort_values("clinical_rank").reset_index(drop=True)

disc_with = build_discordance("with_weights")
disc_uni  = build_discordance("uniform")
discordance_all = pd.concat([d for d in (disc_with, disc_uni) if not d.empty],
                            ignore_index=True) if (not disc_with.empty or not disc_uni.empty) else pd.DataFrame()
if not discordance_all.empty:
    discordance_all.to_csv(os.path.join(OUT_DIR, "clinical_discordance.csv"),
                           index=False)


# ======================================================================
# ANALYSIS 4 : leave-one-gene-out sensitivity
# ======================================================================
def leave_out_effect(mode, gene):
    """Return per-trait dict of {'baseline_rank', 'new_rank', shifts...} for all drugs."""
    use_w = (mode == "withW")
    frames = []
    for trait in TRAITS:
        ldf = long_store.get((mode, trait))
        if ldf is None:
            continue
        c_full = reconstruct_contributions(ldf, use_weights=use_w)
        if c_full.empty:
            continue
        base_full = drug_scores_from_contrib(c_full)
        c_drop = c_full[c_full["gene"] != gene]
        base_drop = drug_scores_from_contrib(c_drop)
        if base_full.empty or base_drop.empty:
            continue
        m = base_full[["drug", "rank"]].merge(
            base_drop[["drug", "rank"]], on="drug",
            suffixes=("_full", "_drop"))
        m["rank_shift"] = m["rank_drop"] - m["rank_full"]  # +ve = becomes safer
        m["trait"] = trait
        frames.append(m)
    if not frames:
        return pd.DataFrame()
    out = pd.concat(frames, ignore_index=True)
    return out

sensitivity_store = {}   # (gene, mode) -> agg df
for gene in LEAVE_OUT_GENES:
    for mode in ("withW", "noW"):
        eff = leave_out_effect(mode, gene)
        if eff.empty:
            continue
        agg = (eff.groupby("drug")
                  .agg(mean_shift=("rank_shift", "mean"),
                       n_traits=("trait", "nunique"))
                  .reset_index())
        agg["abs_shift"] = agg["mean_shift"].abs()
        agg = agg.sort_values("abs_shift", ascending=False)
        sensitivity_store[(gene, mode)] = agg
        agg.to_csv(os.path.join(
            OUT_DIR, f"leaveout_{gene}_{'withW' if mode=='withW' else 'uniform'}.csv"),
            index=False)


# ======================================================================
# ANALYSIS 5 : drugs most sensitive to TWAS scaling (V1 vs V4)
# ======================================================================
scaling_rows = []
for mode in ("withW", "noW"):
    for trait in TRAITS:
        v4 = risk_store.get(("V4", mode, trait))
        v1 = risk_store.get(("V1", mode, trait))
        if v4 is None or v1 is None:
            continue
        m = v4[["drug_key", "drug", "risk_rank"]].merge(
            v1[["drug_key", "risk_rank"]], on="drug_key",
            suffixes=("_V4", "_V1"))
        # +ve => TWAS scaling made it riskier (rank number dropped)
        m["twas_rank_effect"] = m["risk_rank_V1"] - m["risk_rank_V4"]
        m["mode"] = "with_weights" if mode == "withW" else "uniform"
        m["trait"] = trait
        scaling_rows.append(m)

scaling_df = pd.concat(scaling_rows, ignore_index=True) if scaling_rows else pd.DataFrame()
scaling_agg = pd.DataFrame()
if not scaling_df.empty:
    scaling_df.to_csv(os.path.join(OUT_DIR, "twas_scaling_sensitivity.csv"),
                      index=False)
    scaling_agg = (scaling_df.groupby(["mode", "drug"])["twas_rank_effect"]
                   .mean().reset_index()
                   .rename(columns={"twas_rank_effect": "mean_twas_rank_effect"}))
    scaling_agg["abs_effect"] = scaling_agg["mean_twas_rank_effect"].abs()


# ======================================================================
# ANALYSIS 6 : receptor co-occurrence in top high-risk drugs
# ======================================================================
def cooccurrence(mode, trait, top_drugs=TOP_DRUGS_NET):
    rd = risk_store.get(("V4", "withW" if mode == "withW" else "noW", trait))
    ldf = long_store.get((mode, trait))
    if rd is None or ldf is None:
        return pd.DataFrame()
    top = set(rd.nsmallest(top_drugs, "risk_rank")["drug"])
    sub = ldf[ldf["drug"].isin(top) & ldf["receptor"].notna()]
    pair_counts = {}
    for drug, grp in sub.groupby("drug"):
        recs = sorted(grp["receptor"].dropna().unique())
        for a, b in itertools.combinations(recs, 2):
            key = (a, b)
            pair_counts[key] = pair_counts.get(key, 0) + 1
    if not pair_counts:
        return pd.DataFrame()
    rows = [{"receptor_a": a, "receptor_b": b, "n_drugs": n}
            for (a, b), n in pair_counts.items()]
    return pd.DataFrame(rows).sort_values("n_drugs", ascending=False)

cooc_pooled = {}
for mode in ("withW", "noW"):
    frames = []
    for trait in TRAITS:
        c = cooccurrence(mode, trait)
        if not c.empty:
            c["trait"] = trait
            frames.append(c)
    if frames:
        allc = pd.concat(frames, ignore_index=True)
        pooled = (allc.groupby(["receptor_a", "receptor_b"])["n_drugs"]
                  .sum().reset_index().sort_values("n_drugs", ascending=False))
        cooc_pooled[mode] = pooled
        pooled.to_csv(os.path.join(
            OUT_DIR, f"receptor_cooccurrence_{'withW' if mode=='withW' else 'uniform'}.csv"),
            index=False)


# ======================================================================
# RECONSTRUCTION VALIDATION vs saved risk CSVs
# ======================================================================
valid_rows = []
for mode in ("withW", "noW"):
    use_w = (mode == "withW")
    for trait in TRAITS:
        ldf = long_store.get((mode, trait))
        saved = risk_store.get(("V4", mode, trait))
        if ldf is None or saved is None:
            continue
        recon = drug_scores_from_contrib(
            reconstruct_contributions(ldf, use_weights=use_w))
        if recon.empty:
            continue
        recon["drug_key"] = recon["drug"].astype(str).str.strip().str.lower()
        m = recon[["drug_key", "rank"]].merge(
            saved[["drug_key", "risk_rank"]], on="drug_key")
        if len(m) >= 3 and m["rank"].nunique() > 1 and m["risk_rank"].nunique() > 1:
            if _HAVE_SCIPY:
                rho = float(spearmanr(m["rank"], m["risk_rank"])[0])
            else:
                rho = float(m["rank"].corr(m["risk_rank"], method="spearman"))
        else:
            rho = np.nan
        valid_rows.append({"mode": "with_weights" if use_w else "uniform",
                           "trait": trait, "n": len(m),
                           "spearman_recon_vs_saved": rho})
validation_df = pd.DataFrame(valid_rows)
if not validation_df.empty:
    validation_df.to_csv(os.path.join(OUT_DIR, "reconstruction_validation.csv"),
                         index=False)


# ======================================================================
# WRITE DETAILED SUMMARY
# ======================================================================
def _fmt(x, spec=".3f"):
    try:
        if x is None or (isinstance(x, float) and np.isnan(x)):
            return "NA"
        return format(x, spec)
    except Exception:
        return str(x)

L, bar, sub = [], "=" * 82, "-" * 82
L.append(bar)
L.append("N05A Ki→DDD→TWAS — STAGE 2 MECHANISTIC DOWNSTREAM ANALYSIS")
L.append(bar)
L.append(f"Generated            : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
L.append(f"Pipeline output root : {PIPE_ROOT}")
L.append(f"Analysis output dir  : {OUT_DIR}")
L.append(f"Traits               : {', '.join(TRAITS)}")
L.append(f"scipy: {_HAVE_SCIPY}  |  matplotlib: {_HAVE_MPL}")
L.append("")
L.append("Reconstruction formula (mirrors pipeline V4 logp_expo):")
L.append("  affinity   = log1p(max(affinity_dose_score,0))")
L.append(f"  twas_scale = exp({LOGP_BETA} * min(-log10 p, {MAX_NEGLOG10_P}))")
L.append("  weight     = literature weight (with-weights) or 1.0 (uniform)")
L.append("  contribution = affinity * weight * twas_scale")
L.append(f"  drug score = Σcontribution × (1+{SECONDARY_BOOST}·max|z|) × "
         f"(1+{TWAS_SIG_BONUS}·n_sig), normalised to {REF_DRUG}=100")
L.append("")

# --- 0. file discovery ---
L.append(bar); L.append("0. INPUT FILE DISCOVERY"); L.append(bar)
n_ok = sum(1 for r in load_log if r[4] == "OK")
L.append(f"Located OK: {n_ok} / {len(load_log)}")
for ver, mode, trait, kind, status, path in load_log:
    if status != "OK":
        L.append(f"  [{status:<16}] {ver} {mode:<5} {trait:<7} {kind:<5} -> {path}")
L.append("")

# --- validation ---
L.append(bar); L.append("RECONSTRUCTION VALIDATION (recon rank vs saved risk_rank)"); L.append(bar)
if validation_df.empty:
    L.append("  No overlap available to validate.")
else:
    L.append(f"  {'mode':<16}{'trait':<8}{'n':>4}{'spearman':>12}")
    L.append("  " + sub)
    for _, r in validation_df.iterrows():
        L.append(f"  {r['mode']:<16}{r['trait']:<8}{int(r['n']):>4}"
                 f"{_fmt(r['spearman_recon_vs_saved']):>12}")
    mean_rho = validation_df["spearman_recon_vs_saved"].mean()
    L.append(f"  mean Spearman(recon, saved) = {_fmt(mean_rho)} "
             f"(≈1.0 confirms faithful reconstruction)")
L.append("")

# --- 1. global receptor contribution ---
L.append(bar); L.append("1. GLOBAL PER-RECEPTOR CONTRIBUTION (pooled across traits)"); L.append(bar)
if global_pooled.empty:
    L.append("  No contribution data.")
else:
    for mode_label in ("with_weights", "uniform"):
        g = (global_pooled[global_pooled["mode"] == mode_label]
             .sort_values("contribution", ascending=False).head(TOP_RECEPTORS))
        if g.empty:
            continue
        total = global_pooled.loc[global_pooled["mode"] == mode_label,
                                  "contribution"].sum()
        L.append(f"### {mode_label.upper()}")
        L.append(f"    {'#':>2}  {'gene':<10}{'contribution':>14}{'%_of_total':>12}")
        for i, (_, r) in enumerate(g.iterrows(), 1):
            pct = 100 * r["contribution"] / total if total else np.nan
            L.append(f"    {i:>2}  {str(r['gene'])[:9]:<10}"
                     f"{_fmt(r['contribution'],'.4g'):>14}{_fmt(pct,'.1f'):>12}")
        L.append("")
    L.append("  Interpretation: with-weights concentrates signal in H1/HTR2C/CHRM3;")
    L.append("  uniform spreads it, letting high-affinity broad binders + strong-TWAS")
    L.append("  receptors (e.g. adrenergic) rise in relative importance.")
L.append("")

# --- 2. TWAS vs literature weight ---
L.append(bar); L.append("2. TWAS SIGNAL vs LITERATURE WEIGHT  (novel-mechanism candidates)"); L.append(bar)
if twas_weight_df.empty:
    L.append("  No TWAS/gene data available.")
else:
    L.append(f"  Top receptor genes by max |TWAS z| across traits:")
    L.append(f"    {'gene':<10}{'max|z|':>9}{'peak_trait':>12}{'lit_weight':>12}")
    for _, r in twas_weight_df.head(TOP_RECEPTORS).iterrows():
        L.append(f"    {str(r['gene'])[:9]:<10}{_fmt(r['max_abs_twas_z'],'.2f'):>9}"
                 f"{str(r['trait']):>12}{_fmt(r['literature_weight'],'.2f'):>12}")
    L.append("")
    L.append(f"  ** NOVEL-MECHANISM CANDIDATES (|z|>={NOVEL_Z_MIN} AND "
             f"lit_weight<={NOVEL_W_MAX}):")
    if novel_candidates.empty:
        L.append("       (none passed both thresholds)")
    else:
        for _, r in novel_candidates.iterrows():
            L.append(f"       {str(r['gene']):<10} |z|={_fmt(r['max_abs_twas_z'],'.2f')} "
                     f"(peak {r['trait']}), lit_weight={_fmt(r['literature_weight'],'.2f')}")
    L.append("")
    L.append("  These receptors carry strong lipid genetic signal but were assigned")
    L.append("  low a-priori metabolic weight → candidate under-recognised pathways.")
L.append("")

# --- 3. clinical discordance ---
L.append(bar); L.append("3. DISCORDANCE vs CLINICAL METABOLIC-LIABILITY LITERATURE"); L.append(bar)
L.append("  (model_rank = mean risk_rank pooled across traits, re-ranked;")
L.append("   discordance = model_rank − clinical_rank; +ve = model ranks LOWER risk)")
if discordance_all.empty:
    L.append("  No overlap with clinical reference list.")
else:
    for mode_label, dsub in (("with_weights", disc_with), ("uniform", disc_uni)):
        if dsub.empty:
            continue
        L.append(f"### {mode_label.upper()}")
        L.append(f"    {'drug':<16}{'clin':>6}{'model':>7}{'discord':>9}")
        for _, r in dsub.iterrows():
            L.append(f"    {str(r['drug'])[:15]:<16}{int(r['clinical_rank']):>6}"
                     f"{int(r['model_rank']):>7}{int(r['discordance']):>+9}")
        # biggest disagreements
        big = dsub.reindex(dsub["discordance"].abs().sort_values(ascending=False).index)
        L.append("    Largest disagreements:")
        for _, r in big.head(4).iterrows():
            direction = "model UNDER-rates" if r["discordance"] > 0 else "model OVER-rates"
            L.append(f"       {r['drug']}: {direction} risk "
                     f"(Δ={int(r['discordance']):+d})")
        L.append("")
L.append("")

# --- 4. leave-one-gene-out ---
L.append(bar); L.append("4. LEAVE-ONE-GENE-OUT SENSITIVITY (mean rank shift; +ve = drug becomes SAFER)"); L.append(bar)
if not sensitivity_store:
    L.append("  No sensitivity data.")
else:
    for gene in LEAVE_OUT_GENES:
        for mode in ("withW", "noW"):
            agg = sensitivity_store.get((gene, mode))
            if agg is None or agg.empty:
                continue
            mode_label = "with_weights" if mode == "withW" else "uniform"
            movers = agg[agg["abs_shift"] > 0].head(8)
            L.append(f"### Remove {gene}  [{mode_label}]")
            if movers.empty:
                L.append("    (no drug changed rank)")
            else:
                L.append(f"    {'drug':<20}{'mean_shift':>11}{'n_traits':>10}")
                for _, r in movers.iterrows():
                    L.append(f"    {str(r['drug'])[:19]:<20}"
                             f"{_fmt(r['mean_shift'],'+.1f'):>11}"
                             f"{int(r['n_traits']):>10}")
            L.append("")
    L.append("  ADRB1 focus: large shifts here indicate the strong β1-adrenergic lipid")
    L.append("  TWAS signal is a genuine driver rather than a robustness artefact.")
L.append("")

# --- 5. TWAS scaling sensitivity ---
L.append(bar); L.append("5. DRUGS MOST SENSITIVE TO TWAS SCALING (V1 no-scaling → V4 logp_expo)"); L.append(bar)
L.append("  effect = rank_V1 − rank_V4 (pooled mean over traits; +ve = scaling made it RISKIER)")
if scaling_agg.empty:
    L.append("  V1 and/or V4 risk CSVs unavailable — cannot compute.")
else:
    for mode_label in ("with_weights", "uniform"):
        s = (scaling_agg[scaling_agg["mode"] == mode_label]
             .sort_values("abs_effect", ascending=False).head(12))
        if s.empty:
            continue
        L.append(f"### {mode_label.upper()}")
        L.append(f"    {'drug':<20}{'mean_effect':>12}")
        for _, r in s.iterrows():
            L.append(f"    {str(r['drug'])[:19]:<20}"
                     f"{_fmt(r['mean_twas_rank_effect'],'+.1f'):>12}")
        L.append("")
L.append("")

# --- 6. receptor co-occurrence ---
L.append(bar); L.append(f"6. RECEPTOR CO-OCCURRENCE IN TOP-{TOP_DRUGS_NET} HIGH-RISK DRUGS (pooled)"); L.append(bar)
if not cooc_pooled:
    L.append("  No co-occurrence data.")
else:
    for mode in ("withW", "noW"):
        pooled = cooc_pooled.get(mode)
        if pooled is None or pooled.empty:
            continue
        mode_label = "with_weights" if mode == "withW" else "uniform"
        L.append(f"### {mode_label.upper()}  — most frequent receptor pairs")
        L.append(f"    {'receptor A':<12}{'receptor B':<12}{'n_drug-traits':>14}")
        for _, r in pooled.head(12).iterrows():
            L.append(f"    {str(r['receptor_a'])[:11]:<12}"
                     f"{str(r['receptor_b'])[:11]:<12}{int(r['n_drugs']):>14}")
        L.append("")
L.append("")

# --- 7. headline takeaways ---
L.append(bar); L.append("7. HEADLINE TAKEAWAYS (auto-generated)"); L.append(bar)
if not global_pooled.empty:
    for mode_label in ("with_weights", "uniform"):
        g = (global_pooled[global_pooled["mode"] == mode_label]
             .sort_values("contribution", ascending=False).head(3))
        if not g.empty:
            genes = ", ".join(g["gene"].astype(str))
            L.append(f"  * {mode_label}: top contributing receptors = {genes}.")
if not novel_candidates.empty:
    genes = ", ".join(novel_candidates["gene"].astype(str).head(6))
    L.append(f"  * Novel-mechanism candidates (high TWAS, low literature weight): {genes}.")
if not discordance_all.empty and not disc_with.empty:
    worst = disc_with.reindex(disc_with["discordance"].abs()
                              .sort_values(ascending=False).index).iloc[0]
    L.append(f"  * Largest clinical discordance (with-weights): {worst['drug']} "
             f"(Δ={int(worst['discordance']):+d} vs clinical rank).")
for gene in ("ADRB1",):
    agg = sensitivity_store.get((gene, "withW"))
    if agg is not None and not agg.empty:
        top = agg.iloc[0]
        L.append(f"  * Removing {gene} most affects '{top['drug']}' "
                 f"(mean rank shift {_fmt(top['mean_shift'],'+.1f')}).")
if not validation_df.empty:
    L.append(f"  * Reconstruction fidelity: mean Spearman = "
             f"{_fmt(validation_df['spearman_recon_vs_saved'].mean())} vs saved scores.")
L.append("")

# --- artefacts ---
L.append(bar); L.append("8. GENERATED ARTEFACTS"); L.append(bar)
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*.csv"))):
    L.append(f"  [CSV] {f}")
for f in sorted(glob.glob(os.path.join(FIG_DIR, "*.png"))):
    L.append(f"  [FIG] {f}")
L.append("")
L.append(bar); L.append("END OF STAGE 2 SUMMARY"); L.append(bar)

summary_text = "\n".join(L)
summary_path = os.path.join(OUT_DIR, "downstream_stage2_summary.txt")
with open(summary_path, "w", encoding="utf-8") as fh:
    fh.write(summary_text)

print(summary_text)
print("\n" + "#" * 82)
print(f"# Stage 2 summary written to: {summary_path}")
print(f"# All CSVs + figures in:      {OUT_DIR}")
print("#" * 82)

# (optional) copy back to Drive – uncomment if mounted
# !cp -r "/content/pipeline_output/downstream_stage2" "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS/"

N05A Ki→DDD→TWAS — STAGE 2 MECHANISTIC DOWNSTREAM ANALYSIS
Generated            : 2026-07-15 14:42:37
Pipeline output root : /content/pipeline_output
Analysis output dir  : /content/pipeline_output/downstream_stage2
Traits               : ldl, hdl, logtg, nonhdl, tc
scipy: True  |  matplotlib: True

Reconstruction formula (mirrors pipeline V4 logp_expo):
  affinity   = log1p(max(affinity_dose_score,0))
  twas_scale = exp(0.045 * min(-log10 p, 25.0))
  weight     = literature weight (with-weights) or 1.0 (uniform)
  contribution = affinity * weight * twas_scale
  drug score = Σcontribution × (1+0.02·max|z|) × (1+0.1·n_sig), normalised to chlorpromazine=100

0. INPUT FILE DISCOVERY
Located OK: 30 / 30

RECONSTRUCTION VALIDATION (recon rank vs saved risk_rank)
  mode            trait      n    spearman
  ----------------------------------------------------------------------------------
  with_weights    ldl       52       1.000
  with_weights    hdl       52       1.000
  with_weights    

In [ ]:
!cp "/content/pipeline_output" "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS" -r

# The End